<a href="https://colab.research.google.com/github/BrunaFerreira/Mestrado_UNIFESP/blob/main/Revisao_2_Avaliacao_Abstracts_Lista_Artigos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Base Completa de Artigos**

In [ ]:
!pip install Unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
from unidecode import unidecode
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
resultado = pd.read_csv(path_to + '4_Lista_Intermediaria_Artigos_202607.csv')
pubmed = pd.read_csv(path + 'papers_pubmed_2018_Jun2026.csv')
scopus = pd.read_csv(path + 'papers_scopus_2018_Jun2026.csv')

In [ ]:
# Base de Dados sem repetidos
resultado  = resultado[resultado['Repetidos']!=1]

In [ ]:
## Quantidade artigos não repetidos
resultado.shape[0]

461

In [ ]:
resultado.Status_Final.value_counts()

,count
Status_Final,
0.0,263
1.0,149
-1.0,49


# **Step 1: Avaliar Abstract de Novos Artigos**

In [ ]:
resultado[resultado['Status_Final'] == -1].sort_values('Nome').head(26)

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
79,10.3390/healthcare14101401,_0080,9.0,Scopus,41.0,A REVIEW OF DATA ENGINEERING IN UNITED STATES ...,Revisão,0,-1.0
210,10.1111/exsy.70271,_0211,9.0,Scopus,172.0,A REVIEW OF FEDERATED LEARNING UNDER DATA HETE...,Revisão,0,-1.0
45,10.11834/jig.250168,_0046,9.0,Scopus,7.0,A SURVEY OF MULTIMODAL EMOTION RECOGNITION FRO...,NaN,0,-1.0
220,10.1007/978-3-032-23176-5_13,_0221,9.0,Scopus,182.0,A SYSTEMATIC MACHINE LEARNING APPROACH FOR BRA...,NaN,0,-1.0
143,10.7717/peerj-cs.3905,_0144,9.0,Scopus,105.0,A SYSTEMATIC REVIEW AND UNIFIED MULTI PERSPECT...,NaN,0,-1.0
134,10.3390/app16105018,_0135,9.0,Scopus,96.0,A SYSTEMATIC REVIEW OF TRANSFORMER BASED MODEL...,Revisão,0,-1.0
411,10.1109/DASA68193.2025.11499090,_0412,11.0,Scopus,147.0,ADVANCEMENTS IN IMAGE CAPTIONING: A COMPREHENS...,NaN,0,-1.0
116,10.1007/978-981-95-3978-9_1,_0117,9.0,Scopus,78.0,ADVANCING WELDING DEFECT DETECTION IN MARITIME...,NaN,0,-1.0
249,10.1515/9783112214374-022,_0250,9.0,Scopus,211.0,AI BIAS AND FAIRNESS: UNVEILING SOLUTIONS FOR ...,NaN,0,-1.0
145,10.1109/IIPEM65914.2025.11548455,_0146,9.0,Scopus,107.0,AI NATIVE 6G NETWORKS: ADAPTIVE RESOURCE ALLOC...,NaN,0,-1.0


In [ ]:
df = resultado[resultado['Status_Final'] == -1]
df = df[['Codigo','DOI','Query','Nome','Base']]
df.shape

(49, 5)

In [ ]:
colunas =  ['DOI','Abstract','Query', 'Artigo', 'Base', 'Cod','Nome', 'Codigo']
pubmed_abs = pubmed[[ 'doi','abstract', 'query', 'artigo', 'base', 'cod', 'Title', 'Codigo']]
pubmed_abs.columns = colunas
scopus_abs = scopus[['DOI','Abstract','Query', 'Artigo', 'Base', 'Cod','Title', 'Codigo']]
scopus_abs.columns = colunas

base = pd.concat([pubmed_abs, scopus_abs], ignore_index=True)
base['DOI'] = base['DOI'].fillna('-1').astype(str)

In [ ]:
df = df.merge(
    base,
    on=['DOI','Query','Base','Nome'],
    how='left',
    suffixes=('', '_Anteriores')
)

In [ ]:
df.head()

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
0,_0010,10.1016/j.banm.2026.01.021,6.0,APPLICATIONS OF ARTIFICIAL INTELLIGENCE TO THE...,Scopus,Predictive models based on artificial intellig...,10.0,0.0,10
1,_0046,10.11834/jig.250168,9.0,A SURVEY OF MULTIMODAL EMOTION RECOGNITION FRO...,Scopus,Emotion recognition is an essential research d...,7.0,0.0,46
2,_0073,10.14132/j.cnki.1673-5439.2022.03.011,9.0,FCAT FL: AN EFFICIENT FEDERATED LEARNING ALGOR...,Scopus,Aiming at the problem that non-independent and...,34.0,0.0,73
3,_0076,10.4238/q3b3py28,9.0,SECURE VISION: REAL TIME AI SURVEILLANCE FRAME...,Scopus,Deep learning is moving toward automating inte...,37.0,0.0,76
4,_0077,10.1038/s44401-026-00097-w,9.0,TRANSFERRING HEALTHCARE RISK PREDICTION MODELS...,Scopus,Cross-state deployment of Medicaid risk predic...,38.0,0.0,77


In [ ]:
def avaliacao_artigo (resultado, DOI, Status, Criterio):
  # Mostrar o texto do Abstract
  print = resultado[resultado['DOI']==DOI]
  resultado.loc[
      (resultado["DOI"] == DOI) ,
       ["Status_Final", "Criterio_Final"]] = [Status, Criterio]
  print = pd.concat([print, resultado[resultado['DOI']==DOI]], ignore_index=True)
  return print, resultado

In [ ]:
def avaliacao_artigo_nome (resultado,Nome, status,criterio):
  # Máscara do filtro
  mask = (
      resultado["Nome"].str.contains(Nome, na=False, regex=False) &
      (resultado['Status_Final'] == -1)
  )

  # Salva o estado antes da alteração
  antes = resultado.loc[mask].copy()

  # Atualiza o DataFrame
  resultado.loc[mask, ["Status_Final", "Criterio_Final"]] = [status, criterio]

  # Estado após a alteração
  depois = resultado.loc[
      resultado["Nome"].str.contains(Nome, na=False, regex=False) &
      resultado["Criterio_Final"].notna()
  ].copy()

  # Junta antes e depois
  comparacao = pd.concat([antes, depois], ignore_index=True)

  return comparacao, resultado



In [ ]:
df = df.sort_values('Nome')
df.shape

(49, 9)

In [ ]:
#print(
#    df.loc[df['DOI'] == '10.1016/j.banm.2026.01.021', 'Abstract'].iloc[0]
#)

### **Preenchendo Artigos Novos**

In [ ]:
resultado[resultado['Status_Final'] == -1]['DOI']

,DOI
9,10.1016/j.banm.2026.01.021
45,10.11834/jig.250168
72,10.14132/j.cnki.1673-5439.2022.03.011
75,10.4238/q3b3py28
76,10.1038/s44401-026-00097-w
79,10.3390/healthcare14101401
88,10.56294/dm2025658
102,10.1007/s11263-026-02864-6
116,10.1007/978-981-95-3978-9_1
134,10.3390/app16105018


In [ ]:
resultado.Status_Final.value_counts()

,count
Status_Final,
0.0,263
1.0,149
-1.0,49


### **Artigo 1**
```
Abstract

DOI: '10.1111/exsy.70271'
```

Federated learning (FL) has emerged as an impactful paradigm for privacy-preserving machine learning, and allows model training without the need to share raw data. However, data heterogeneity across clients challenges practical FL deployment. Data space heterogeneity and statistical heterogeneity create significant training difficulties. System heterogeneity imposes additional external constraints. These combined factors impair convergence and reduce model performance. They also raise concerns regarding fairness, scalability and robustness. Focused on data heterogeneity, this review provides a structured analysis of FL. It encompasses three key areas: core categorizations of data heterogeneity, algorithmic advances (e.g., personalized FL, mixture-of-experts architectures, transfer learning-based solutions) and system-level techniques spanning communication optimization, resource adaptation and secure collaboration. We further synthesize benchmark efforts and real-world applications in healthcare, finance, nuclear power and the Internet of Things (IoT)/edge computing to highlight the practical implications of heterogeneity-aware FL. Finally, we identify key challenges and outline promising research directions towards scalable, fair and adaptive FL systems capable of operating in complex real-world settings. This survey aims to serve as a reference point and conceptual roadmap for future research in heterogeneous FL. © 2026 John Wiley & Sons Ltd.




In [ ]:
DOI= '10.1111/exsy.70271'
status  = 0
criterio = 'Aprendizado Federado'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1111/exsy.70271,_0211,9.0,Scopus,172.0,A REVIEW OF FEDERATED LEARNING UNDER DATA HETE...,Revisão,0,-1.0
1,10.1111/exsy.70271,_0211,9.0,Scopus,172.0,A REVIEW OF FEDERATED LEARNING UNDER DATA HETE...,Aprendizado Federado,0,0.0


In [ ]:
resultado.Status_Final.value_counts()

,count
Status_Final,
0.0,264
1.0,149
-1.0,48


### **Artigo 2**
```
DOI : 10.11834/jig.250168
```
Emotion recognition is an essential research direction in artificial intelligence\(AI\),focusing on modeling the relationship between emotional expressions and measurable features\. It enables computers to recognize and understand human emotions,thereby playing a crucial role in various domains of human-computer interaction\. From intelligent assistants to mental health monitoring and social robotics,emotion-aware systems are becoming increasingly pervasive,making affective computing a cornerstone of future intelligent technologies\. Psychological studies have long confirmed that human emotions are typically expressed through a combination of behavioral cues,most notably facial expressions,speech prosody,and language content\. These multimodal signals are often intertwined,with each modality providing complementary information that reflects the inner emotional state of an individual\. In addition to these observable behaviors,physiological signals such as electroencephalogram and electrocardiogram also vary with emotional changes\. However,collecting physiological signals often requires contact-based or invasive sensors,limiting their practical use in daily environments\. By contrast,facial expressions,voice,and language are naturally occurring,easily accessible,and can be captured unobtrusively by cameras and microphones\. These advantages make non-invasive emotion recognition approaches particularly attractive for real-world applications,where user comfort and ease of deployment are critical\. The field of noninvasive emotion recognition has made significant progress in recent years,supported by advancements in deep learning,multimodal fusion techniques,and the increasing availability of emotion-labeled datasets\. At the theoretical level,emotions are typically represented through two main models:discrete and dimensional\. The discrete model classifies affect into a set of basic categories such as happiness,sadness,anger,fear,and surprise\. This model is intuitive and aligns well with how humans often perceive emotion\. Alternatively,the dimensional model maps emotions onto a continuous space,commonly defined by valence\(positive to negative\),arousal\(calm to excited\),and sometimes dominance\(submissive to dominant\)\. This model allows for more nuanced and dynamic descriptions of emotional states\. Both models form the basis for designing annotation protocols and developing machine learning algorithms for emotion detection\. A fundamental driver of research in this field is the availability of multimodal emotional datasets\. Numerous public databases support facial expression recognition,speech emotion analysis,and affective text mining\. For example,facial datasets like AffectNet and real-world affective faces database\(RAF-DB\)contain millions of annotated images representing a broad spectrum of expressions,while audio datasets like interactive emotional dyadic motion capture database\(IEMOCAP\)and Ryerson audiovisual database of emotional speech and song\(RAVDESS\)offer rich emotional speech recordings\. Language datasets,including those developed for sentiment analysis challenges,provide labeled corpora for detecting emotion in both written and spoken text\. Multimodal datasets like Carnegie Mellon University Multimodal Opinion Sentiment and Emotion Intensity \(CMU-MOSI\)and Carnegie Mellon University Multimodal Opinion Sentiment and Emotion Intensity\(CMU-MOSEI\)integrate synchronized recordings of facial expressions,voice,and transcribed language,supporting more comprehensive studies on multimodal fusion and cross-modal emotion reasoning\. Technological progress in this area has been fueled by innovations in machine learning and signal processing\. Facial expression recognition has evolved from handcrafted feature extraction to end-to-end deep learning frameworks using convolutional neural networks\(CNNs\),attention mechanisms,and,more recently,vision transformers\. Temporal modeling techniques have also been used to capture dynamic facial cues,including micro-expressions that may reveal concealed emotions\. In speech emotion recognition,acoustic features such as pitch,intensity,Mel-frequency cepstral coefficients\(MFCCs\),and spectral entropy are used to infer emotion\. Recurrent neural networks\(RNNs\),convolutional layers,and transformers have also shown strong performance,especially when trained on large emotional corpora\. For language-based emotion analysis,both traditional natural language processing \(NLP\)techniques and modern pre-trained language models like bidirectional encoder representations from Transformers \(BERT\)and generative pretrained Transformer\(GPT\)have proven effective in extracting emotional content from text,including subtle and context-dependent cues\. A major trend in the field is the integration of multiple modalities to improve recognition accuracy and robustness\. Multimodal emotion recognition systems aim to leverage complementary information from facial,vocal,and linguistic cues\. To achieve this,various fusion strategies have been developed:early fusion,which combines raw features;late fusion,which merges decision outputs;and hybrid fusion methods,which integrate features at intermediate stages\. Cross-modal attention,contrastive learning,and reinforcement learning have been used to address issues such as modality imbalance,missing data,and asynchronous signals\. Therefore,multimodal approaches consistently outperform single-modality systems,especially in real-world conditions where one modality might be unreliable\. The applications of emotion recognition are also expanding into several domains\. In healthcare,non-invasive emotion recognition is used for monitoring mental health,detecting depression,and supporting emotional therapy\. In education,it enables intelligent tutoring systems to gauge student engagement and adapt teaching strategies\. In intelligent driving,it helps assess driver fatigue and stress to enhance road safety\. Customer service chatbots and entertainment platforms are also increasingly equipped with emotion-aware capabilities to personalize interactions\. The ability to recognize and adapt to human emotions not only improves user satisfaction but also fosters more natural and effective communication between humans and machines\. Despite these advances,the field of emotion recognition continues to face numerous challenges\. Cultural and individual differences in emotional expression can hinder the generalizability of models trained on specific datasets\. Emotions are often ambiguous and context-dependent,making accurate annotation and recognition difficult\. Real-time deployment requires models that are accurate,lightweight,and efficient\. Moreover,ethical concerns surrounding emotion recognition—especially regarding privacy,consent,and potential misuse of affective data—are increasingly coming into focus\. Thus,emotion-aware systems must be developed and deployed responsibly,with transparency and fairness\. Several promising directions are emerging\. Looking ahead,transfer learning and domain adaptation techniques can help models generalize better across diverse user groups and environments\. Furthermore,self-supervised and unsupervised learning approaches may alleviate the dependency on large annotated datasets\. Researchers are also exploring explainable AI methods to increase the interpretability and trustworthiness of emotion recognition systems\. Additionally,the fusion of modalities through generative techniques and cross-modal alignment is opening new avenues for robust and flexible emotion understanding,especially in noisy or incomplete input scenarios\. In conclusion,this review presents a comprehensive analysis of non-invasive emotion recognition based on facial expressions,speech,and language\. By summarizing theoretical foundations,available resources,technical advancements,practical applications,and ongoing challenges,it aims to provide researchers and practitioners with a clear understanding of the current landscape and future potential of this rapidly evolving field\. © 2025 Editorial and Publishing Board of JIG\. All rights reserved\.|




In [ ]:
DOI  = '10.11834/jig.250168'
status = 0
criterio = 'Realizar uma revisão completa do estado da arte em reconhecimento não invasivo de emoções utilizando expressões faciais, fala e linguagem, destacando fundamentos teóricos, métodos, aplicações, desafios e perspectivas futuras.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.11834/jig.250168,_0046,9.0,Scopus,7.0,A SURVEY OF MULTIMODAL EMOTION RECOGNITION FRO...,NaN,0,-1.0
1,10.11834/jig.250168,_0046,9.0,Scopus,7.0,A SURVEY OF MULTIMODAL EMOTION RECOGNITION FRO...,Realizar uma revisão completa do estado da art...,0,0.0


### **Artigo 3**
```
DOI : 10.14132/j.cnki.1673-5439.2022.03.011
```
Aiming at the problem that non-independent and identically distributed \(Non-IID\) data affects the convergence speed, fairness and accuracy of federated learning, a fast and fair federated migration learning framework—FCAT-FL based on Non-IID data is proposed\. This framework improves the traditional federated learning strategy of weighing the contribution of aggregation according to the proportion of client data, and assigns adaptive weights to clients in each round of aggregation dynamically according to the relationship between client model parameters and server model parameters\. And a personalized migration learning model and a momentum gradient descent algorithm are introduced in the client to speed up the convergence of the local model\. The experimental results show that, compared with several baseline aggregation strategies, when the data of some clients is Non-IID, the global iteration round of strategy 1 in FCAT-FL is reduced and the fairness and the accuracy among clients are improved\. Moreover, the use of migration learning reduces the number of model parameters that clients need to train and upload\. Therefore, FCAT-FL is suitable for mobile edge networks with limited client resources\. © 2022 Journal of Nanjing Institute of Posts and Telecommunications\. All rights reserved\.





In [ ]:
DOI = '10.14132/j.cnki.1673-5439.2022.03.011'
status = 0
criterio = 'Aprendizado Federado . Desenvolver um framework de aprendizado federado rápido, justo e eficiente para ambientes com dados Non-IID, melhorando a convergência, a precisão e a equidade entre clientes, enquanto reduz o consumo de recursos computacionais.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.14132/j.cnki.1673-5439.2022.03.011,_0073,9.0,Scopus,34.0,FCAT FL: AN EFFICIENT FEDERATED LEARNING ALGOR...,NaN,0,-1.0
1,10.14132/j.cnki.1673-5439.2022.03.011,_0073,9.0,Scopus,34.0,FCAT FL: AN EFFICIENT FEDERATED LEARNING ALGOR...,Aprendizado Federado . Desenvolver um framewor...,0,0.0


### **Artigo 4**
```
DOI : 10.56294/dm2025658
```
Deep Learning is a rapidly evolving field with critical contributions to various domains including security, healthcare, and human — computer interaction, etc\. It reviews the significant developments in the area of facial recognition using deep learning techniques\. It explains deep learning models such as Convolutional Neural Networks \(CNNs\), Recurrent Neural Networks \(RNNs\), Long Short-Term Memory Networks \(LSTMs\), and Generative Adversarial Networks \(GANs\), as well as hybrid models and transfer learning uses\. It also addresses technical, ethical, and legal challenges that arise for facial analysis systems and emphasizes the need for real-time processing, multi-modal systems, and robust algorithms to improve the technical accuracy and fairness of facial analysis\. © 2025; Los autores\.





In [ ]:
DOI = '10.56294/dm2025658'
status = 0
criterio = 'Reconhecimento Facial. Analisar e revisar os avanços do Deep Learning aplicados ao reconhecimento facial, discutindo modelos, aplicações, desafios e perspectivas para tornar esses sistemas mais precisos, robustos e justos.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.56294/dm2025658,_0089,9.0,Scopus,50.0,SYSTEMATIC REVIEW: RECENT ADVANCEMENTS IN DEEP...,Revisão,0,-1.0
1,10.56294/dm2025658,_0089,9.0,Scopus,50.0,SYSTEMATIC REVIEW: RECENT ADVANCEMENTS IN DEEP...,Reconhecimento Facial. Analisar e revisar os a...,0,0.0


### **Artigo 5**
```
DOI : 10.1007/978-981-95-3978-9_1
```
Weld defect detection is crucial for ensuring the safety and reliability of piping systems in the oil and gas industry, especially in challenging marine and offshore environments\. Traditional non-destructive testing \(NDT\) methods often fail to detect subtle or internal defects, leading to potential failures and costly downtime\. Furthermore, existing neural network-based approaches for defect classification frequently rely on arbitrarily selected pre-trained architectures and lack interpretability, raising safety concerns for deployment\. To address these challenges, this paper introduces “Adapt-WeldNet,” an adaptive framework for welding defect detection that systematically evaluates various pre-trained architectures, transfer learning strategies, and adaptive optimizers to identify the best performing model and hyperparameters, optimizing defect detection and providing actionable insights\. Additionally, a novel Defect Detection Interpretability Analysis \(DDIA\) framework is proposed to enhance system transparency\. DDIA employs Explainable AI \(XAI\) techniques, such as Grad-CAM and LIME, alongside domain-specific evaluations validated by certified ASNT NDE Level II professionals\. Incorporating a Human-in-the-Loop \(HITL\) approach and aligning with the principles of Trustworthy AI, DDIA ensures the reliability, fairness, and accountability of the defect detection system, fostering confidence in automated decisions through expert validation\. By improving both performance and interpretability, this work enhances trust, safety, and reliability in welding defect detection systems, supporting critical operations in offshore and marine environments\. © The Author\(s\), under exclusive license to Springer Nature Singapore Pte Ltd\. 2026\.





In [ ]:
DOI = '10.1007/978-981-95-3978-9_1'
status = 0
criterio = 'detecção de defeitos em soldagem que avalia sistematicamente diferentes arquiteturas pré-treinadas, estratégias de aprendizado por transferência (transfer learning) e otimizadores adaptativos, a fim de identificar o modelo e os hiperparâmetros de melhor desempenho'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1007/978-981-95-3978-9_1,_0117,9.0,Scopus,78.0,ADVANCING WELDING DEFECT DETECTION IN MARITIME...,NaN,0,-1.0
1,10.1007/978-981-95-3978-9_1,_0117,9.0,Scopus,78.0,ADVANCING WELDING DEFECT DETECTION IN MARITIME...,detecção de defeitos em soldagem que avalia si...,0,0.0


### **Artigo 6**
```
DOI : 10.3390/app16105018
```
Depression is a critical global public health challenge, and the demand for accurate automated detection methods has generated considerable research interest in Transformer-based models\. Despite their substantial promise, a comprehensive investigation into their architectural efficacy, intrinsic mechanisms, and barriers to practical implementation remains lacking\. Following the Preferred Reporting Items for Systematic Reviews and Meta-Analyses \(PRISMA\) 2020 guidelines, this systematic review was conducted across six databases \(IEEE Xplore, Elsevier, Springer, MDPI, PubMed, and arXiv\)\. The final search was performed in October 2025, covering English-language empirical studies published between 2020 and 2025 that employed Transformer-based architectures for depression detection\. Risk of bias and methodological quality were independently appraised by two authors using a six-dimension structured rubric, with disagreements resolved by a third author\. Findings were narratively synthesized given substantial cross-study heterogeneity\. This systematic review analyzed 46 studies and provided the first comprehensive, mechanism-level, architecturally stratified comparison of encoder-only, decoder-only, hybrid, and multimodal fusion paradigms, examining self-attention dynamics and transfer learning strategies\. Since 2019, these frameworks have evolved from text-centric approaches to advanced multimodal systems\. Encoder-only models show consistently strong results in high-throughput text-based screening, decoder-only models demonstrate stronger few-shot learning capabilities, hybrid architectures show the highest observed median performance in clinical interview settings across the reviewed studies, and multimodal fusion systems offer complementary advantages when heterogeneous signal integration is critical\. These trends are task-contextualized and should not be interpreted as unconditional rankings, given heterogeneity in evaluation metrics and tasks across studies\. Nonetheless, four principal challenges hinder clinical translation: overreliance on self-reported data, cross-linguistic bias, absence of uncertainty quantification, and substantial computational overhead\. Future efforts should shift from incremental benchmark improvements toward clinical utility through standardized psychiatric validation, uncertainty-aware architectures, fairness-enforced training across diverse populations, and the integration of Transformer-based models with wearable and mobile health data to improve detection stability and reduce translational risk\. This systematic review was registered on the Open Science Framework \(OSF; DOI: 10\.17605/OSF\.IO/SYF9N\)\. This research was funded by the Faculty of Information Science and Technology and by Universiti Kebangsaan Malaysia under Grant TAP-K014364\. © 2026 by the authors\.

In [ ]:
DOI = '10.3390/app16105018'
status  = 0
criterio = 'Revisão sistemática sobre depressao, examinou 46 estudos e forneceu a primeira comparação abrangente, em nível de mecanismo e estratificada por arquitetura, entre paradigmas de modelos somente codificadores (encoder-only), somente decodificadores (decoder-only), híbridos e de fusão multimodal, analisando tanto as dinâmicas de autoatenção (self-attention) quanto as estratégias de aprendizado por transferência (transfer learning).'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.3390/app16105018,_0135,9.0,Scopus,96.0,A SYSTEMATIC REVIEW OF TRANSFORMER BASED MODEL...,Revisão,0,-1.0
1,10.3390/app16105018,_0135,9.0,Scopus,96.0,A SYSTEMATIC REVIEW OF TRANSFORMER BASED MODEL...,"Revisão sistemática sobre depressao, examinou ...",0,0.0


### **Artigo 7**
```
DOI : 10.1109/iCONECCT67014.2025.11469951
```
Across the world, skin-related disorders represent one of the most frequently occurring health problems, affecting individuals in all age groups and contributing significantly to the global burden of diseases\. In recent years, techniques such as machine learning \(ML\) and deep learning \(DL\) have emerged as powerful tools to support dermatologists in the detection and classification of skin conditions\. This paper reviews recent literature on the applications of ML in skin disease diagnosis and classification\. It highlights approaches such as convolutional neural networks, transfer learning, hybrid models, etc\., outlining their advantages and limitations in clinical practice\. Popular datasets, including ISIC, HAM10000, PH2, and DermNet, are discussed with respect to their size, diversity, and annotation quality\. Commonly adopted evaluation metrics - accuracy, precision, recall, F1-score, AUC, and Cohen's kappa - are also examined as benchmarks of model performance\. Despite rapid progress, challenges such as data variability, limited generalizability, interpretability, and fairness remain barriers to real-world adoption\. Promising research directions include explainable AI, federated learning, multimodal integration, and fairness-aware modeling\. © 2025 IEEE\.




In [ ]:
DOI = '10.1109/iCONECCT67014.2025.11469951'
status = 1
criterio = 'Revisão abrangente das técnicas de Machine Learning e Deep Learning aplicadas ao diagnóstico de doenças de pele, analisando métodos, bases de dados, métricas de avaliação, desafios atuais e perspectivas futuras para adoção clínica'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/iCONECCT67014.2025.11469951,_0148,9.0,Scopus,109.0,MACHINE LEARNING FOR SKIN DISEASE DIAGNOSIS: A...,NaN,0,-1.0
1,10.1109/iCONECCT67014.2025.11469951,_0148,9.0,Scopus,109.0,MACHINE LEARNING FOR SKIN DISEASE DIAGNOSIS: A...,Revisão abrangente das técnicas de Machine Lea...,0,1.0


### **Artigo 8**
```
DOI : 10.1016/j.banm.2026.01.021
```
Predictive models based on artificial intelligence (AI) are transforming the search for new treatments for complex chronic conditions such as Sjögren's disease or Systemic Lupus Erythematosus. Computational models of these autoimmune diseases are built from patient molecular profiling data obtained through multiomics technologies integrated by AI. These analyses help to represent patient heterogeneity and to identify relevant therapeutic targets among the molecular pathways dysregulated in these diseases. AI is also used to identify and optimize drug candidates that interact with these therapeutic targets, and to design combined therapies. Cohorts of virtual patients can be created to predict in silico the efficacy of drug candidates. By stratifying patients into molecularly defined subgroups — thus enabling optimized therapeutic options — AI is fostering a computational precision medicine that could ultimately link detailed individual patient characteristics with the predicted properties of billions of drug candidates with the aim to propose increasingly personalized treatments. © 2026 l'Académie nationale de médecine; Les modèles prédictifs basés sur l'intelligence artificielle (IA) transforment la recherche de nouveaux traitements contre des pathologies chroniques complexes telles que la maladie de Sjögren, ou le lupus érythémateux disséminé. Des modèles computationnels de ces maladies auto-immunes sont établis à partir de données de profilage moléculaire multi-omiques des patients intégrées par l'IA. Ces analyses permettent de représenter l'hétérogénéité des patients et d'identifier des cibles thérapeutiques pertinentes parmi les voies moléculaires dérégulées dans ces maladies. L'IA est également utile pour identifier et optimiser des candidats-médicaments interagissant avec ces cibles thérapeutiques, mais aussi pour concevoir des thérapies combinées. Des cohortes de patients virtuels peuvent être créées pour prédire in silico l'efficacité des médicaments à l’échelle de patients individuels. En stratifiant les patients en sous-groupes définis moléculairement pour mieux adapter les traitements, l'IA permet d'envisager à terme une médecine de précision computationnelle susceptible de mettre en relation les caractéristiques individuelles des patients avec les propriétés prédites de milliards de candidats-médicaments, afin de proposer des traitements de plus en plus personnalisés. © 2026 l'Académie nationale de médecine





In [ ]:
DOI = '10.1016/j.banm.2026.01.021'
status = 1
criterio = 'utilizar a inteligência artificial e dados multiômicos para desenvolver uma medicina de precisão personalizada, capaz de prever quais tratamentos serão mais eficazes para cada paciente com doenças autoimunes complexas.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1016/j.banm.2026.01.021,_0010,6.0,Scopus,10.0,APPLICATIONS OF ARTIFICIAL INTELLIGENCE TO THE...,Revisão,0,-1.0
1,10.1016/j.banm.2026.01.021,_0010,6.0,Scopus,10.0,APPLICATIONS OF ARTIFICIAL INTELLIGENCE TO THE...,utilizar a inteligência artificial e dados mul...,0,1.0


### **Artigo 9**
```
DOI : 10.1109/OJCOMS.2026.3693148
```
High-altitude platform stations \(HAPS\) offer a pragmatic backbone for extending 6G non-terrestrial networks \(NTNs\) into dense urban cores, sparsely connected remote regions, and disaster-stricken areas\. In these environments, cognitive radio networks \(CRNs\) must arbitrate spectrum access without trusting all agents or relying on intact terrestrial backhaul\. Accordingly, in the present study, we propose a federated hierarchical reinforcement learning \(F-HRL\) framework where secondary users \(SUs\) cooperatively access primary users’ \(PUs\) spectrum, ground next-generation Node Bs \(gNBs\) and unmanned aerial vehicles \(UAVs\) enforce trust-aware scheduling, while HAPS coordinators perform Byzantine-resilient aggregation using satellite backhaul\. Area-specific reward shaping allows the same policy stack to prioritize spectral efficiency in urban deployments, coverage in remote regions, and rapid recovery after infrastructure loss\. We benchmark F-HRL against non-cognitive baselines, simple heuristics, and the hierarchical deep reinforcement learning framework across the following three key scenarios over 5 000 decision steps: nominal spectrum sharing, disaster recovery with a gNB outage, and adversarial environments with malicious SUs\. Experimental results demonstrate that F-HRL achieves a nearly 8-fold higher spectrum efficiency than the non-cognitive operation, maintains 86% SU quality-of-service \(QoS\) satisfaction with a Jain fairness index of 0\.950, and sustains over 95% of nominal throughput during infrastructure failures\. Under attack conditions with 25% compromised agents, trust-weighted aggregation successfully isolates malicious updates while preserving legitimate throughput\. These findings demonstrate that HAPS-centric F-HRL provides adaptive, secure, and resilient spectrum management for future NTN deployments\. © 2020 IEEE\.





In [ ]:
DOI = '10.1109/OJCOMS.2026.3693148'
status = 0
criterio = 'Sistema inteligente para gerenciamento de espectro em redes 6G não terrestres'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/OJCOMS.2026.3693148,_0212,9.0,Scopus,173.0,FEDERATED HIERARCHICAL REINFORCEMENT LEARNING ...,NaN,0,-1.0
1,10.1109/OJCOMS.2026.3693148,_0212,9.0,Scopus,173.0,FEDERATED HIERARCHICAL REINFORCEMENT LEARNING ...,Sistema inteligente para gerenciamento de espe...,0,0.0


### **Artigo 10**
```
DOI : 10.1007/978-3-032-23176-5_13
```
A significant challenge in brain cancer diagnosis in Ecuador is the reliance on visual interpretation of magnetic resonance imaging \(MRI\) by specialists, a process that is time-consuming and prone to human error\. Alternatively, biopsies are invasive and costly, limiting their accessibility\. In this study, the CRISP-DM methodology is applied to develop a Deep Learning-based classification model to predict the presence of malignant tumors in MRI\. The phases of the proposed method are: 1\. Data Preparation Phase; the Brats 2023 Adult Glioma dataset is utilized for cancer patients, and synthetic samples are generated using data augmentation techniques to represent non-cancer patients, achieving a balanced dataset to promote fairness\. It is worth highlighting that the models developed are custom-built and trained from scratch, without the use of transfer learning\. 2\. Classification Model Development Phase; three models are developed: Convolutional Neural Network \(CNN\), Residual Neural Network \(ResNet\), and Support Vector Machine \(SVM\)\. 3\. Evaluation Phase; the models are evaluated using classical classification metrics: accuracy, precision, recall, and F1-score\. The CNN achieved an accuracy of 99\.77%, outperforming models such as SVM and ResNet\. This work lays the groundwork for future research involving local datasets with images from Ecuadorian patients, both with and without cancer, to enhance the model’s generalization and applicability in real clinical settings, while also emphasizing the importance of fairness in diagnostic outcomes\. © The Author\(s\), under exclusive license to Springer Nature Switzerland AG 2026\.





In [ ]:
DOI = '10.1007/978-3-032-23176-5_13'
status = 1
criterio = 'Desenvolver um sistema baseado em Deep Learning para detectar tumores cerebrais malignos em imagens de ressonância magnética de forma automática e precisa.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1007/978-3-032-23176-5_13,_0221,9.0,Scopus,182.0,A SYSTEMATIC MACHINE LEARNING APPROACH FOR BRA...,NaN,0,-1.0
1,10.1007/978-3-032-23176-5_13,_0221,9.0,Scopus,182.0,A SYSTEMATIC MACHINE LEARNING APPROACH FOR BRA...,Desenvolver um sistema baseado em Deep Learnin...,0,1.0


### **Artigo 11**
```
DOI : 10.3390/s26103215
```

Accurate and consistent forensic bruise assessment is critical in ensuring positive clinical and legal outcomes for victims of violence\. In this study, a framework for automated bruise detection is presented that, for the first time, integrates narrowband alternate-light-source \(ALS\) forensic imaging and ambient white light imaging\. This evaluation framework is designed to address long-standing issues with respect to equitable performance across skin tones and lighting scenarios via a combination of novel model diagnostic strategies\. In particular, skin-tone balancing during training and testing, threshold-sensitivity analysis, and embedding-similarity partitioning are employed to quantify the model robustness and deployment trade-offs that arise in forensic image analysis\. Models were implemented with ImageNet-pretrained backbones and trained on a unique, multi-annotator full-consensus dataset comprising both white-light and ALS \(415 nm and 450 nm\) images\. The protocol emphasizes three axes of operational relevance: \(1\) illumination composition in training \(W/ALS ratio\); \(2\) subgroup fairness via targeted balancing; and \(3\) model operating-point selection \(confidence and IoU thresholds\) informed by confidence-stability metrics and bootstrapped uncertainty estimates\. Systematic W/ALS ratio sweeps indicate peak accuracy under ALS-dominant training and declining performance as the proportion of white-light images increases within the training set\. Skin-tone balancing reduced failure rates for darker skin tones but increased overprediction in some demographic subgroups\. Embedding-similarity and seen/unseen injury analyses demonstrate inflated generalization under image-level partitioning\. Ultimately, the findings suggest that future researchers and developers should employ injury-level data partitioning and ensure a weighted balance of ALS images during training\. © 2026 by the authors\.




In [ ]:
DOI = '10.3390/s26103215'
status = 1
criterio = 'Desenvolver e avaliar um sistema automatizado de detecção de hematomas que: Utilize imagens forenses avançadas; Funcione de forma mais justa para diferentes tons de pele; Seja robusto em diferentes condições de iluminação.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.3390/s26103215,_0234,9.0,Scopus,195.0,EVALUATION FRAMEWORK FOR BRUISE DETECTION: SYS...,NaN,0,-1.0
1,10.3390/s26103215,_0234,9.0,Scopus,195.0,EVALUATION FRAMEWORK FOR BRUISE DETECTION: SYS...,Desenvolver e avaliar um sistema automatizado ...,0,1.0


### **Artigo 12**
```
DOI : 10.12688/f1000research.166307.2
```
Background: Early and accurate prediction of Ischemic Heart Disease (IHD) is critical to reducing cardiovascular mortality through timely intervention. While deep learning (DL) models have shown promise in disease prediction, many lack interpretability, generalizability, and fairness-particularly when deployed across demographically diverse populations. These shortcomings limit clinical adoption and risk reinforcing healthcare disparities. Methods: This study proposes a novel model: X-TLRABiLSTM (Explainable Transfer Learning-based Residual Attention Bidirectional LSTM). The architecture integrates transfer learning from pre-trained cardiovascular models into a BiLSTM framework with residual attention layers to improve temporal feature extraction and convergence. To ensure transparency, the model incorporates SHAP (SHapley Additive exPlanations) to quantify the contribution of each clinical feature to the final prediction. Additionally, a demographic reweighting strategy is applied to the training process to reduce bias across subgroups defined by age, gender, and ethnicity. The model was evaluated on the UCI Heart Disease dataset using 10-fold cross-validation. Results: The X-TLRABiLSTM model achieved a classification accuracy of 98.2%, with an F1-score of 98.1% and an AUC of 99.1%, outperforming standard ML classifiers and state-of-the-art DL baselines. SHAP-based interpretability analysis highlighted clinically relevant predictors such as chest pain type, ST depression, and thalassemia. A fairness-aware reweighting strategy was applied during training, and fairness evaluation revealed minimal performance disparity across demographic subgroups, with F1-score gaps ≤ 0.6% and error rate gaps ≤ 0.4%. Confusion matrix analysis demonstrated low false-positive and false-negative rates, reinforcing the model's reliability for clinical deployment. Conclusions: X-TLRABiLSTM offers a highly accurate, interpretable, and demographically fair framework for IHD prognosis. By combining transfer learning, residual attention, explainable AI, and fairness-aware optimization, this model advances trustworthy AI in healthcare. Its successful performance on benchmark clinical data supports its potential for real-world integration in ethical, AI-assisted cardiovascular diagnostics. Copyright: © 2025 D C et al.







In [ ]:
DOI = '10.12688/f1000research.166307.2'
status = 1
criterio  = 'Desenvolver um modelo de IA para prognóstico de doença cardíaca que combine: Alto desempenho; Transparência; Justiça entre diferentes grupos populacionais.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.12688/f1000research.166307.2,_0239,9.0,Scopus,200.0,EXPLAINABLE TRANSFER LEARNING WITH RESIDUAL AT...,NaN,0,-1.0
1,10.12688/f1000research.166307.2,_0239,9.0,Scopus,200.0,EXPLAINABLE TRANSFER LEARNING WITH RESIDUAL AT...,Desenvolver um modelo de IA para prognóstico d...,0,1.0


### **Artigo 13**
```
DOI : 10.1109/ICCSC67078.2026.11468610
```
Ride-hailing apps (e.g., Uber, Ola) have revolutionized mobility in urban areas but present special challenges for Indian cities such as Bangalore. We survey the latest research on forecasting and pricing in ride-hailing, with an emphasis on demand forecasting, trip price prediction, and dynamic surge pricing. We emphasize approaches from traditional time-series (ARIMA) to contemporary deep learning (CNNs, GNNs, Transformers, RL) for modeling intricate spatiotemporal dynamics. India-specific issues receive special focus: absence of trip-level open data (only aggregate travel times through Uber Movement), hyper-spatial heterogeneity, regulatory surge-price limits, and fairness/equity issues. We recognize gaps (e.g., fairness, privacy, transfer learning across cities) and suggest cutting-edge ML directions (e.g., graph neural networks, attention models, multi-task learning, pricing reinforcement learning, federated learning, data fusion with weather/events). As an example, we examine publicly disclosed Uber Movement travel-time data for Bangalore to compute travel-time isochrones and make educated guesses about probable surge areas. We are including comparative analyses highlight model performance and demand patterns to substantiate arguments. Our critique incorporates more than 25 recent studies and regulatory documents, utilizing recent studies and regulatory documents, to guide ML and transport researchers targeting Indian ride-hailing. © 2026 IEEE.





In [ ]:
DOI = '10.1109/ICCSC67078.2026.11468610'
status = 0
criterio = 'Revisão abrangente das técnicas de previsão e precificação em aplicativos de transporte por aplicativo, destacando desafios específicos da Índia e propondo novas direções de pesquisa em aprendizado de máquina.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/ICCSC67078.2026.11468610,_0255,9.0,Scopus,216.0,"FARE PREDICTION, DEMAND FORECASTING, AND SURGE...",NaN,0,-1.0
1,10.1109/ICCSC67078.2026.11468610,_0255,9.0,Scopus,216.0,"FARE PREDICTION, DEMAND FORECASTING, AND SURGE...",Revisão abrangente das técnicas de previsão e ...,0,0.0


### **Artigo 14**
```
DOI : 10.1201/9781003737810-7
```
Foundation models are gaining popularity in healthcare due to their ability to support diverse downstream tasks such as diagnosis, treatment recommendations, medical image analysis, patient monitoring, and personalized medicine. With their growing influence comes an urgent need for responsible development and deployment, encompassing fairness, transparency, explainability, alignment, security, privacy, ethics, safety, and regulations. Properly managing these factors has a significant impact on patient care and medical outcomes. This chapter reviews the challenges and opportunities within responsible foundation models for healthcare, synthesizing perspectives from academia, industry, policymakers, and healthcare organizations. The future of the responsible foundation model in healthcare depends on embedding responsibility into both design and practice. Such practice highlights that the trajectory of healthcare foundation models will be determined not only by technical sophistication but by the extent to which they serve public interest, equity, and trust. © 2027 selection and editorial matter, Christo El Morr, Rachel da Silveira Gorman, Elham Dolatabadi, and Laleh Seyyed-Kalantari; individual chapters, the contributors.






In [ ]:
DOI = '10.1201/9781003737810-7'
status = 0
criterio = 'Foundations Models. Discutir como os foundation models podem ser desenvolvidos e utilizados de forma responsável na saúde, garantindo benefícios clínicos, éticos e sociais, ao mesmo tempo em que promovem confiança, equidade e segurança. '

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1201/9781003737810-7,_0266,11.0,Scopus,1.0,RESPONSIBLE FOUNDATION MODELS FOR HEALTHCARE: ...,NaN,0,-1.0
1,10.1201/9781003737810-7,_0266,11.0,Scopus,1.0,RESPONSIBLE FOUNDATION MODELS FOR HEALTHCARE: ...,Foundations Models. Discutir como os foundatio...,0,0.0


### **Artigo 15**
```
DOI : 10.1007/978-3-032-18782-6_2
```
Breakthroughs in artificial intelligence (AI) have transformed diagnostics in cognitive health, particularly within the realms of neuroimaging. New technologies, such as Generative Adversarial Networks (GANs) and Variational Autoencoders (VAEs), have made it possible to detect previously subclinical changes in patients long before the associated changes become symptomatic, as well as synthesise and correct medical images. The diagnostic tools provide an unparalleled accuracy of anomalies at the tissue level, which permits timely and personalised interventions using multimodal data, including genetics, imaging, and behavioural data. AI improves diagnostic precision, thus enabling cross-population generalisability, which addresses dataset imbalance through synthesis, and augments clinician trust through explainable interfaces. However, the implementation of such systems is not without challenges, such as a disparity in diagnostic criteria (e.g., DSM-5 vs. ICD-10/11), deficits in longitudinal samples, and structural biases within the training data. Technical innovations need to be matched with evolution in ethics, law, and regulation to safeguard patients’ data and autonomy while ensuring fairness in the access and distribution of care. Federated systems and learning ‘human-in-the-loop’ models may mitigate this problem by enabling shared-ownership model construction across multiple health systems, while maintaining clinician supervision of model and system deployment. This chapter brings together the clinical, technical, and ethical considerations of AI-assisted cognitive diagnosis. It underscores the importance of standardization regarding differing diagnostic processes in different locations, interprofessional collaboration, and continuous cross-validation for the reproducible, transparent, and accountable incorporation of AI. Ultimately, the fusion of medicine and technology offers the prospect of a paradigm shift in cognitive healthcare, emphasizing the shift toward proactive and individualized approaches. This fusion could transform the diagnosis, evaluation, and treatment processes of cognitive disorders. © The Author(s), under exclusive license to Springer Nature Switzerland AG 2026.






In [ ]:
DOI = '10.1007/978-3-032-18782-6_2'
status = 0
criterio = ' Analisar como a Inteligência Artificial está transformando o diagnóstico de distúrbios cognitivos, abordando simultaneamente aspectos clínicos, técnicos e éticos. GANs (Generative Adversarial Networks) Geração de imagens médicas sintéticas. Correção e aprimoramento de imagens. Detecção precoce de alterações cerebrais. VAEs (Variational Autoencoders) Modelagem de padrões complexos. Reconstrução e síntese de imagens médicas. Apoio à identificação de alterações subclínicas.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1007/978-3-032-18782-6_2,_0294,11.0,Scopus,29.0,FUNDAMENTALS OF COGNITIVE DISEASES AND DIAGNOS...,NaN,0,-1.0
1,10.1007/978-3-032-18782-6_2,_0294,11.0,Scopus,29.0,FUNDAMENTALS OF COGNITIVE DISEASES AND DIAGNOS...,Analisar como a Inteligência Artificial está ...,0,0.0


### **Artigo 16**
```
DOI : 10.1109/QCNC69040.2026.00172
```
Quantum machine learning has emerged as a promising approach to improve feature extraction and classification tasks in high-dimensional data domains such as medical imaging. In this work, we present a hybrid Quantum-Classical Convolutional Neural Network (QCNN) architecture designed for the binary classification of the BreastMNIST dataset, a standardized benchmark for distinguishing between benign and malignant breast tumors. Our architecture integrates classical convolutional feature extraction with two distinct quantum circuits: an amplitude-encoding variational quantum circuit (VQC) and an angle-encoding VQC circuit with circular entanglement, both implemented on four qubits. These circuits generate quantum feature embeddings that are fused with classical features to form a joint feature space, which is subsequently processed by a fully connected classifier. To ensure fairness, the hybrid QCNN is parameter-matched against a baseline classical CNN, allowing us to isolate the contribution of quantum layers. Both models are trained under identical conditions using the Adam optimizer and binary cross-entropy loss. Experimental evaluation in five independent runs demonstrates that the hybrid QCNN achieves statistically significant improvements in classification accuracy compared to the classical CNN, as validated by a one-sided Wilcoxon signed rank test ({p=0.03125) and supported by large effect size of Cohen's d = 2.14. Our results indicate that hybrid QCNN architectures can leverage entanglement and quantum feature fusion to enhance medical image classification tasks. This work establishes a statistical validation framework for assessing hybrid quantum models in biomedical applications and highlights pathways for scaling to larger datasets and deployment on near-term quantum hardware. © 2026 IEEE.





In [ ]:
DOI = '10.1109/QCNC69040.2026.00172'
status = 0
criterio = 'Investigar se a computação quântica pode melhorar a classificação de imagens médicas em comparação com redes neurais convencionais.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/QCNC69040.2026.00172,_0301,11.0,Scopus,36.0,PARALLEL MULTI CIRCUIT QUANTUM FEATURE FUSION ...,NaN,0,-1.0
1,10.1109/QCNC69040.2026.00172,_0301,11.0,Scopus,36.0,PARALLEL MULTI CIRCUIT QUANTUM FEATURE FUSION ...,Investigar se a computação quântica pode melho...,0,0.0


### **Artigo 17**
```
DOI : 10.1109/IC3ECSBHI67834.2026.11468880
```
Medical image segmentation models are mostly confronted with class imbalance, where the class is much smaller compared to the background. This results in poor and biased results. The evaluation of medical image segmentation models in class imbalance scenarios is still a challenging problem. The loss functions, such as Dice Loss, are efficient in handling class overlaps but are poor in handling boundary errors. In this paper, a novel concept called the 'Multi-Metric Adaptive Loss' function is introduced, which combines the three aspects of class overlap, boundary errors, and size-based fairness in the model's learning process. The proposed 'Multi-Metric Adaptive Loss' function was implemented and tested on the Kvasir-Seg 'Polyp Segmentation' and 'MoNuSeg - Nuclei Segmentation' datasets, which are size and instance imbalanced. The proposed function has shown better performance compared to the existing 'Dice Loss' function and the combination of 'Dice Loss' and 'CrossEntropy Loss' functions. The proposed function has shown a 18.5 percent improvement in 95 percent Hausdorff Distance for small nuclei in MoNuSeg and high 'Dice' scores, as the proposed function is closely related to the evaluation metrics, resulting in more reliable results in small lesion segmentation in medical images. © 2026 IEEE.





In [ ]:
DOI = '10.1109/IC3ECSBHI67834.2026.11468880'
status = 0
criterio = 'Desenvolver uma função de perda mais eficaz para segmentação de imagens médicas desbalanceadas, produzindo resultados mais precisos e confiáveis, especialmente para objetos pequenos e difíceis de segmentar.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/IC3ECSBHI67834.2026.11468880,_0316,11.0,Scopus,51.0,MULTI METRIC ADAPTIVE LOSS MMAL FUNCTION FOR O...,NaN,0,-1.0
1,10.1109/IC3ECSBHI67834.2026.11468880,_0316,11.0,Scopus,51.0,MULTI METRIC ADAPTIVE LOSS MMAL FUNCTION FOR O...,Desenvolver uma função de perda mais eficaz pa...,0,0.0


### **Artigo 18**
```
DOI : 10.1145/3793542
```
Ensuring equitable Artificial Intelligence (AI) in healthcare demands systems that make unbiased decisions across all demographic groups, bridging technical innovation with ethical principles. Foundation Models (FMs), trained on vast datasets through self-supervised learning, enable efficient adaptation across medical imaging tasks while reducing dependency on labeled data. These models demonstrate potential for enhancing fairness, though significant challenges remain in achieving consistent performance across demographic groups. Our review indicates that effective bias mitigation in FMs requires systematic interventions throughout all stages of development. While previous approaches focused primarily on model-level bias mitigation, our analysis reveals that fairness in FMs requires integrated interventions throughout the development pipeline, from data documentation to deployment protocols. This comprehensive framework advances current knowledge by demonstrating how systematic bias mitigation, combined with policy engagement, can effectively address both technical and institutional barriers to equitable AI in healthcare. The development of equitable FMs represents a critical step toward democratizing advanced healthcare technologies, particularly for underserved populations and regions with limited medical infrastructure and computational resources. © 2026 Copyright held by the owner/author(s).






In [ ]:
DOI = '10.1145/3793542'
status = 1
criterio = 'Desenvolver e promover o uso de Foundation Models equitativos na saúde, garantindo que os benefícios da IA sejam distribuídos de forma justa entre diferentes populações e contextos socioeconômicos.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1145/3793542,_0340,11.0,Scopus,75.0,FAIR FOUNDATION MODELS FOR MEDICAL IMAGE ANALY...,Revisão,0,-1.0
1,10.1145/3793542,_0340,11.0,Scopus,75.0,FAIR FOUNDATION MODELS FOR MEDICAL IMAGE ANALY...,Desenvolver e promover o uso de Foundation Mod...,0,1.0


### **Artigo 19**
```
DOI : 10.1117/12.3085023
```
Fairness and explainability are essential pillars for the development of ethical, trustworthy, and effective AI systems in healthcare. Biases in AI models, particularly those arising from data acquisition, can lead to disparities in clinical outcomes and compromise equity in medical decision-making. While Content-Based Image Retrieval (CBIR) systems offer interpretable visual context to support diagnostic reasoning, they remain vulnerable to dataset-induced biases. In this work, we present the first systematic investigation of covariate bias related to scanning devices in the context of Foundation Models (FMs) for CBIR applications in histopathology, a critical yet previously unexplored dimension in the literature. To facilitate this study, we curated a novel dataset to isolate scanner variability, comprising spatially co-registered image patches obtained from the same histology slides, each scanned using two different whole slide imaging devices. This design ensures confounding factors are held consistent across paired patches. Through a series of experiments across two distinct datasets, we demonstrate that the retrieval performance of state-of-the-art FMs is dependent on scanners, revealing a pronounced lack of robustness to scanner variability. Among the FMs evaluated, iBOT-Path, UNI, Virchow2, and Prov-GigaPath achieved the most stable performance across scanners and datasets. In contrast, KimiaNet, Virchow, Hibou-B, and HIPT have poor performance stability, reflecting lower generalizability. This study provides the first empirical evidence of scanner-induced covariate bias in CBIR applications using FMs, highlighting an important and previously unaddressed challenge in the deployment of trustworthy AI systems for computational pathology. © 2026 COPYRIGHT SPIE.






In [ ]:
DOI = '10.1117/12.3085023'
status = 0
criterio = 'Investigar se diferentes equipamentos de escaneamento de lâminas histológicas introduzem vieses que afetam o desempenho de Foundation Models em sistemas CBIR para histopatologia, avaliando a robustez e a capacidade de generalização desses modelos.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1117/12.3085023,_0369,11.0,Scopus,104.0,INVESTIGATING SCANNER BIAS IN HISTOPATHOLOGY I...,NaN,0,-1.0
1,10.1117/12.3085023,_0369,11.0,Scopus,104.0,INVESTIGATING SCANNER BIAS IN HISTOPATHOLOGY I...,Investigar se diferentes equipamentos de escan...,0,0.0


### **Artigo 20**
```
DOI : 10.18280/isi.310313
```
Keratitis, an inflammation of the cornea, presents significant diagnostic challenges due to overlapping clinical symptoms and the limited availability of expert ophthalmological evaluation, especially in early-stage presentations\. Timely diagnosis is crucial to prevent permanent corneal damage and vision loss\. Clinical datasets used for automated keratitis detection are often highly imbalanced, which can introduce classification bias and impair the generalization capability of deep learning–based models\. To address this issue, synthetic data generation using generative adversarial networks \(GANs\) has been widely adopted to balance minority classes\. However, the effectiveness of such data augmentation in improving both model performance and fairness remains underexplored\. A comparative evaluation was performed using convolutional neural networks \(CNNs\) under two scenarios: \(i\) imbalanced training with real data and \(ii\) balanced training incorporating GAN-generated images\. MobileNet, trained on the imbalanced dataset, achieved the highest accuracy of 94\.53%, demonstrating high sensitivity for the dominant classes\. In contrast, Xception trained on the GAN-balanced dataset exhibited a lower accuracy of 80\.76%, accompanied by inconsistent recall across the target classes, despite the improved class distribution\. Structural similarity index measure \(SSIM\) analysis indicated suboptimal visual fidelity of the synthetic images, particularly for underrepresented categories\. These findings reveal that class balancing via GANs does not consistently enhance classification performance and may introduce representational noise, leading to a trade-off between class balance and overall accuracy\. By emphasizing the importance of jointly evaluating model fairness and the fidelity of GAN-synthesized data, the results highlight the need for more adaptive and quality-aware augmentation strategies in medical image classification tasks\. Copyright © 2026 The authors\. This article is published by IIETA and is licensed under the CC BY 4\.0 license \(http://creativecommons\.org/licenses/by/4\.0/\)\.




In [ ]:
DOI = '10.18280/isi.310313'
status = 1
criterio = 'Investigar se o balanceamento de conjuntos de dados desbalanceados por meio de imagens sintéticas geradas por GANs melhora o desempenho e a equidade de modelos de aprendizado profundo na classificação automática de ceratite'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.18280/isi.310313,_0419,11.0,Scopus,154.0,CRITICAL ASSESSMENT OF DCGAN BASED DATA BALANC...,NaN,0,-1.0
1,10.18280/isi.310313,_0419,11.0,Scopus,154.0,CRITICAL ASSESSMENT OF DCGAN BASED DATA BALANC...,Investigar se o balanceamento de conjuntos de ...,0,1.0


### **Artigo 21**
```
DOI : 10.1201/9781779640710-7
```
Cancer is a global challenge, leading to millions of cases every year\. Uncontrolled cell proliferation and dysfunctional programmed cell death lead to malignant growths called as tumors\. The tumor resection surgery supported with adjuvant radiotherapy and chemotherapy is often a practiced treatment regime to prevent the spread of tumor cells to other parts of the body\. Tumor selectivity is of paramount importance during cancer treatment to prevent damage to healthy tissue and cancer resistance\. The healthcare sector is revolutionized by artificial intelligence \(AI\), and its evolving intersection with robotics has potential to supremely improve the success rate of tumor resection surgery\. Advancements in AI and robotics are improving oncology surgery and have the potential to become completely AI-powered\. AI and robotics-powered semi-automation or automation can help the surgeon in precision, which in turn would result in minimally invasive and standardized surgical procedures\. AI can be used to create 3-dimensional models from medical images and help surgeons with personalized surgical planning\. The current overview highlights the potential of AI within oncology surgery, focusing on its application in preoperative planning, intraoperative guidance, especially for robotic-assisted procedures, and postoperative care and monitoring\. Also, the ethical and regulatory considerations about bias and fairness of AI algorithms, transparency and accountability, data privacy and protection are highlighted with future directions\. © 2026 by Apple Academic Press, Inc\.




In [ ]:
DOI = '10.1201/9781779640710-7'
status = 0
criterio = 'Cirurgia Oncologica. Avaliar e apresentar como a inteligência artificial e a robótica podem aprimorar a cirurgia oncológica em todas as suas etapas — antes, durante e após a operação — considerando também desafios éticos e regulatórios.'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1201/9781779640710-7,_0420,11.0,Scopus,155.0,CLINICAL APPLICATIONS AND CASE STUDIES: EXAMIN...,NaN,0,-1.0
1,10.1201/9781779640710-7,_0420,11.0,Scopus,155.0,CLINICAL APPLICATIONS AND CASE STUDIES: EXAMIN...,Cirurgia Oncologica. Avaliar e apresentar como...,0,0.0


### **Artigo 22**
```
Nome: ALGORITHMS TRAINED ON NORMAL CHEST X RAYS
```
Artificial intelligence is revealing what medicine never intended to encode. Deep vision models, trained on chest X-rays, can now detect not only disease but also invisible traces of social inequality. In this study, we show that state-of-the-art architectures (DenseNet121, SwinV2-T, MedMamba) can predict a patient’s health insurance type, a strong proxy for socioeconomic status, from normal chest X-rays with significant accuracy (AUC ≈ 0.70 on MIMIC-CXR-JPG, 0.68 on CheXpert). The signal was unlikely contributed by demographic features by our machine learning study combining age, race, and sex labels to predict health insurance types. The signal also remains detectable when the model is trained exclusively on a single racial group. Patch-based occlusion reveals that the signal is diffuse rather than localized, embedded in the upper and mid-thoracic regions. This suggests that deep networks may be internalizing subtle traces of clinical environments, equipment differences, or care pathways; learning socioeconomic signals itself. These findings challenge the assumption that medical images are neutral biological data. By uncovering how models perceive and exploit these hidden social signatures, this work reframes fairness in medical AI: the goal is no longer only to balance datasets or adjust thresholds, but to interrogate and disentangle the social fingerprints embedded in clinical data itself. The codes are available at Link. © 2026 CC-BY 4.0, C.-Y. Chen et al.





In [ ]:
# texto = df.loc[
#     df["Nome"].str.contains(
#         'ALGORITHMS TRAINED ON NORMAL CHEST X RAYS CA',
#         na=False
#     )
# ].iloc[0]
# texto

In [ ]:
codigo = 192
Nome = 'ALGORITHMS TRAINED ON NORMAL CHEST X RAYS CA'
Query = 11
status = 1
criterio = ('Mostrar que modelos de inteligência artificial conseguem '
            'extrair informações socioeconômicas ocultas de radiografias '
            'de tórax, levantando importantes questões sobre viés, '
            'equidade e interpretação em IA médica.')

comparacao, resultado  = avaliacao_artigo_nome (resultado,Nome, status,criterio)
comparacao

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,-1,_0468,11.0,Scopus,203.0,ALGORITHMS TRAINED ON NORMAL CHEST X RAYS CAN ...,NaN,0,-1.0
1,-1,_0468,11.0,Scopus,203.0,ALGORITHMS TRAINED ON NORMAL CHEST X RAYS CAN ...,Mostrar que modelos de inteligência artificial...,0,1.0


### **Artigo 23**
```
Nome: EVALUATING THE IMPACT OF MEDICAL IMAGE RECONS
```
AI-based image reconstruction models are increasingly deployed in clinical workflows to improve image quality from noisy data, such as low-dose X-rays or accelerated MRI scans. However, these models are typically evaluated using pixel-level metrics like PSNR, leaving their impact on downstream diagnostic performance and fairness unclear. We introduce a scalable evaluation framework that applies reconstruction and diagnostic AI models in tandem, which we apply to two tasks (classification, segmentation), three reconstruction approaches (U-Net, GAN, diffusion), and two data types (X-ray, MRI) to assess the potential downstream implications of reconstruction. We find that conventional reconstruction metrics poorly track task performance, where diagnostic accuracy remains largely stable even as reconstruction PSNR declines with increasing image noise. Fairness metrics exhibit greater variability, with reconstruction sometimes amplifying demographic biases, particularly regarding patient sex. However, the overall magnitude of this additional bias is modest compared to the inherent biases already present in diagnostic models. To explore potential bias mitigation, we adapt two strategies from classification literature to the reconstruction setting, but observe limited efficacy. Overall, our findings emphasize the importance of holistic performance and fairness assessments throughout the entire medical imaging workflow, especially as generative reconstruction models are increasingly deployed. © 2026 CC-BY 4.0, M. Wohlrapp, N. Bubeck, D. Rueckert & W. Lotter.




In [ ]:
# texto = df.loc[
#      df["Nome"].str.contains(
#          'EVALUATING THE IMPACT OF MEDICAL IMAGE RECONS',
#          na=False
#      )
#  ].iloc[0]
# texto

In [ ]:
Nome = 'EVALUATING THE IMPACT OF MEDICAL IMAGE RECONS'
Query = 11
status = 1
criterio = 'Como modelos de reconstrução de imagens médicas afetam o desempenho diagnóstico e a equidade dos sistemas de IA, indo além de métricas tradicionais como PSNR.'

comparacao, resultado  = avaliacao_artigo_nome (resultado,Nome, status,criterio)
comparacao

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,-1,_0307,11.0,Scopus,42.0,EVALUATING THE IMPACT OF MEDICAL IMAGE RECONST...,NaN,0,-1.0
1,-1,_0307,11.0,Scopus,42.0,EVALUATING THE IMPACT OF MEDICAL IMAGE RECONST...,Como modelos de reconstrução de imagens médica...,0,1.0


### **Artigo 24**
```
Nome: EXPLORING ENTROPY BASED ACTIVE LEARNING FO
```
Active learning (AL) has emerged as a crucial strategy for reducing the prohibitive costs associated with medical image segmentation. However, standard uncertainty-based AL methods typically focus on maximizing performance metrics, ignoring performance disparities or fairness across groups with sensitive attributes. While fair active learning has been explored in classification tasks, its intersection with medical image segmentation remains unaddressed. In this work, we introduced a fairness-aware active learning framework with a Weighted Entropy selection strategy that modulates uncertainty based on current group-specific performance estimates on the labeled set. To decouple true epistemic uncertainty from anatomical volume variances, we further utilized a masked, scaled entropy restricted to the region of interest. The framework was evaluated on synthetic T1-weighted brain MRIs with controlled left caudate bias in both strong and weak bias settings. A 3D U-Net was trained to segment the left caudate under several AL strategies, starting from both demographically balanced and strongly imbalanced initial labeled sets. Experiments demonstrated that our method markedly reduces performance disparities between groups compared to random sampling and standard uncertainty sampling. By prioritizing poorly segmented subgroups during the AL cycles, our method consistently achieved the highest equity-scaled performance and reduced the disparity metric by 75% (strong bias) and 86% (weak bias) relative to standard entropy at the final budget. Overall, this work is among the first studies on fair AL for medical image segmentation, offering an efficient strategy to train more equitable models in resource-constrained environments. © 2026 CC-BY 4.0, G. Danaee, M. Gaillochet, C. Desrosiers, H. Lombaert & S. Bouix.





In [ ]:
# texto = df.loc[
#      df["Nome"].str.contains(
#          'EXPLORING ENTROPY BASED ACTIVE LEARNING FO',
#          na=False
#      )
#  ].iloc[0]
# texto

In [ ]:
Nome = 'EXPLORING ENTROPY BASED ACTIVE LEARNING FO'
Query = 11
status = 0
criterio = 'abordagem de Active Learning para segmentação de imagens médicas que seja capaz de reduzir vieses e desigualdades entre grupos demográficos, ao mesmo tempo em que mantém alta eficiência no uso de dados anotados.'

comparacao, resultado  = avaliacao_artigo_nome (resultado,Nome, status,criterio)
comparacao

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,-1,_0305,11.0,Scopus,40.0,EXPLORING ENTROPY BASED ACTIVE LEARNING FOR FA...,NaN,0,-1.0
1,-1,_0305,11.0,Scopus,40.0,EXPLORING ENTROPY BASED ACTIVE LEARNING FOR FA...,abordagem de Active Learning para segmentação ...,0,0.0


### **Artigo 25**
```
Nome: SECURE VISION: REAL TIME AI SURVEILLANCE FRAMEWORK FOR TACTICAL THREAT RECOGNITION
```
Deep learning is moving toward automating intelligent surveillance and criminal activity detection systems. In this paper, optimized deep learning architectures have been proposed that have been optimized to enhance the detection of crime by facial recognition, real-time video surveillance system, and weapon detection. These systems achieve accuracy, real-time analysis, and scalability through the use of the latest neural network structures, primarily CNNs, RNNs, and hybrid deep learning models. The automated facial recognition systems can track users’ behaviors, register recognized users, recognize suspicious behaviors and detect weapons at video feed frames. Through spatial-temporal analysis, Aides can identify and mitigate response delays in public areas, transport systems and other sensitive locations. Important issues including dataset deficiencies, limits of real-time computing, aggressive attacks and ethical concerns on surveillance AI are addressed.

In [ ]:
texto = df.loc[
     df["Nome"].str.contains(
         'SECURE VISION: REAL TIME AI SURVEILLANCE FRAMEWORK',
         na=False
     )
 ].iloc[0]
texto

,3
Codigo,_0076
DOI,10.4238/q3b3py28
Query,9.0
Nome,SECURE VISION: REAL TIME AI SURVEILLANCE FRAME...
Base,Scopus
Abstract,Deep learning is moving toward automating inte...
Artigo,37.0
Cod,0.0
Codigo_Anteriores,76


In [ ]:
DOI = '10.4238/q3b3py28'
status = 0
criterio = 'detecção de crimes por meio de reconhecimento facial, sistemas de videomonitoramento em tempo real e detecção de armas. Esses sistemas alcançam elevada precisão, análise em tempo real e escalabilidade por meio da utilização das mais recentes arquiteturas de redes neurais'

print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.4238/q3b3py28,_0076,9.0,Scopus,37.0,SECURE VISION: REAL TIME AI SURVEILLANCE FRAME...,NaN,0,-1.0
1,10.4238/q3b3py28,_0076,9.0,Scopus,37.0,SECURE VISION: REAL TIME AI SURVEILLANCE FRAME...,detecção de crimes por meio de reconhecimento ...,0,0.0


### **Artigo 26**
```
Nome: TRANSFERRING HEALTHCARE RISK PREDICTION MODELS BETWEEN MEDICAID POPULATIONS: A TRANSFER LEARNING EVALUATION
```
Cross-state deployment of Medicaid risk prediction models is challenged by demographic, policy, and care-delivery differences that create domain shift. We evaluated transfer learning methods for predicting acute care utilisation between Washington (source; n = 20,744) and Virginia (target; n = 28,901) Medicaid populations enrolled in high-risk care management, where outcome prevalence differed markedly (9.4% vs 25.6%). Nine approaches were compared: source-only and target-only logistic regression, prototypical networks, domain-adversarial neural networks, causal transfer learning, TabTransformer, Enhanced MAML, a meta-ensemble, and a simple average ensemble. On the Virginia hold-out set, the meta-ensemble achieved the highest discrimination (AUC 0.728, 95% CI 0.691–0.764) and best calibration (Brier 0.193, 95% CI 0.180–0.207). Source-only transfer performed similarly (AUC 0.725), with no significant difference (p = 0.454), and both outperformed target-only logistic regression (AUC 0.628; p < 0.001). Enhanced MAML (AUC 0.677) did not improve over naive transfer. Post-hoc isotonic regression substantially improved calibration across models, underscoring the importance of prevalence adjustment. Fairness analyses showed lower race/ethnicity equalized odds differences for target-adapted models (0.132–0.157) than source-only transfer (0.897), though disparities persisted. Findings suggest simple transfer plus recalibration can match complex methods; broader validation is needed. © The Author(s) 2026.

In [ ]:
# DOI  = '10.1038/s44401-026-00097-w'
# df.loc[df['DOI'] == DOI, 'Abstract'].values[0]

In [ ]:
DOI  = '10.1038/s44401-026-00097-w'
Query = 9
status = 1
criterio = 'Avaliar e comparar diferentes métodos de transferência de aprendizado para prever a utilização de cuidados agudos ao transferir modelos de risco do Medicaid entre dois estados dos EUA (Washington e Virgínia), analisando desempenho preditivo, calibração e equidade.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1038/s44401-026-00097-w,_0077,9.0,Scopus,38.0,TRANSFERRING HEALTHCARE RISK PREDICTION MODELS...,NaN,0,-1.0
1,10.1038/s44401-026-00097-w,_0077,9.0,Scopus,38.0,TRANSFERRING HEALTHCARE RISK PREDICTION MODELS...,Avaliar e comparar diferentes métodos de trans...,0,1.0


### **Artigo 27**
```
Nome: A REVIEW OF DATA ENGINEERING IN UNITED STATES HEALTHCARE INFRASTRUCTURE
```

With the rapid advancements in artificial intelligence (AI) and machine learning (ML), the role of data engineering has become increasingly critical due to the growing demands for high-quality, large-scale, and well-structured datasets required to train reliable predictive models. Healthcare is one of the most data-intensive industries and has demonstrated strong potential for AI-driven automation in clinical decision support, diagnostics, and operational efficiency. However, healthcare data is often fragmented across multiple systems, inconsistently formatted, and constrained by privacy and regulatory requirements, creating significant barriers to scalable AI adoption. In this review, we examine recent research on healthcare data engineering and AI applications, focusing on how data pipelines, interoperability, and governance frameworks support or limit real-world deployment. This review examined 68 peer-reviewed studies published between 2018 and 2026 across multiple clinical domains, including oncology, cardiovascular disease, infectious disease, neurological disorders, medical imaging, and algorithmic frameworks for explainability and fairness. The reviewed literature shows that while AI models achieve promising performance across these domains, the lack of standardized data architectures and interoperable infrastructure remains a primary bottleneck. The purpose of this study is to highlight key challenges and emerging solutions in healthcare data engineering and outline the future directions needed to support safe, scalable, and trustworthy AI integration in the United States healthcare system. The intended core contributions of this article are to: (i) identify the need for reliable AI systems for healthcare, (ii) explore challenges associated with implementing AI systems in healthcare from a data engineer’s perspective, and (iii) analyze key limitations of data engineering as it applies to the implementation of AI systems in healthcare. It must be noted that one of the key limitations of this narrative review is that the authors mostly used citations from MDPI journals. © 2026 by the authors.

In [ ]:
DOI  = '10.3390/healthcare14101401'
df.loc[df['DOI'] == DOI]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
5,_0080,10.3390/healthcare14101401,9.0,A REVIEW OF DATA ENGINEERING IN UNITED STATES ...,Scopus,With the rapid advancements in artificial inte...,41.0,0.0,80


In [ ]:
DOI  = '10.3390/healthcare14101401'
Query = 9
status = 0
criterio = 'O objetivo deste estudo é revisar os principais desafios e soluções da engenharia de dados para apoiar a implementação segura, escalável e confiável de sistemas de inteligência artificial na área da saúde, com foco na qualidade dos dados, interoperabilidade e governança.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.3390/healthcare14101401,_0080,9.0,Scopus,41.0,A REVIEW OF DATA ENGINEERING IN UNITED STATES ...,Revisão,0,-1.0
1,10.3390/healthcare14101401,_0080,9.0,Scopus,41.0,A REVIEW OF DATA ENGINEERING IN UNITED STATES ...,O objetivo deste estudo é revisar os principai...,0,0.0


### **Artigo 28**
```
Nome: PARAMETER EFFICIENT FINE TUNING FOR PRE TRAINED VISION MODELS: A SURVEY AND BENCHMARK
```
Pre-trained vision models (PVMs) have demonstrated remarkable adaptability across a wide range of downstream vision tasks, showcasing exceptional performance. However, as these models scale to billions or even trillions of parameters, conventional full fine-tuning has become increasingly impractical due to its high computational and storage demands. To address these challenges, parameter-efficient fine-tuning (PEFT) has emerged as a promising alternative, aiming to achieve performance comparable to full fine-tuning while making minimal adjustments to the model parameters. This paper presents a comprehensive survey of the latest advancements in the visual PEFT field, systematically reviewing current methodologies and categorizing them into four primary categories: addition-based, partial-based, unified-based, and multi-task tuning. In addition, this paper offers an in-depth analysis of widely used visual datasets and real-world applications where PEFT methods have been successfully applied. Furthermore, this paper introduces the V-PEFT Bench, a unified benchmark designed to standardize the evaluation of PEFT methods across a diverse set of vision tasks, ensuring consistency and fairness in comparison. Finally, the paper outlines potential directions for future research to propel advances in the PEFT field. A comprehensive collection of resources and the benchmark codebase are available at https://github.com/synbol/Awesome-Parameter-Efficient-Transfer-Learning. © The Author(s), under exclusive licence to Springer Science+Business Media, LLC, part of Springer Nature 2026.

In [ ]:
DOI  = '10.1007/s11263-026-02864-6'
df.loc[df['DOI'] == DOI]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
7,_0103,10.1007/s11263-026-02864-6,9.0,PARAMETER EFFICIENT FINE TUNING FOR PRE TRAINE...,Scopus,Pre-trained vision models (PVMs) have demonstr...,64.0,0.0,103


In [ ]:
Query = 9
status = 0
criterio = 'O objetivo deste artigo é revisar de forma abrangente os métodos de ajuste fino eficiente em parâmetros (PEFT) para modelos de visão computacional pré-treinados, analisando suas abordagens, aplicações, conjuntos de dados, propondo um benchmark padronizado para comparação e discutindo direções para pesquisas futuras.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1007/s11263-026-02864-6,_0103,9.0,Scopus,64.0,PARAMETER EFFICIENT FINE TUNING FOR PRE TRAINE...,NaN,0,-1.0
1,10.1007/s11263-026-02864-6,_0103,9.0,Scopus,64.0,PARAMETER EFFICIENT FINE TUNING FOR PRE TRAINE...,O objetivo deste artigo é revisar de forma abr...,0,0.0


### **Artigo 29**
```
Nome: CALIBRATED DEEP LEARNING RISK INDEXING AND LATENT BEHAVIOURAL PROFILING FOR OCCUPATIONAL MENTAL HEALTH RISK ASSESSMENT
```
Occupational mental-health risk in knowledge-work settings is an important public-health and psychosocial-support concern because workload demands, career insecurity, limited mentoring, uneven institutional support and barriers to care can increase psychological risk, including in early-career academic environments. Workplace well-being assessments rely on aggregate survey summaries or conventional prediction models, limiting calibration, interpretability, subgroup evaluation and transfer validation. This study develops a computational-intelligence framework for public mental-health decision support using heterogeneous workplace survey data with early-career academics treated as a motivating knowledge-work context rather than as the direct empirical cohort. The proposed approach combines attention-based tabular learning, variational autoencoder latent profiling, stacked ensemble prediction, probability calibration, feature attribution, perturbation analysis, fairness assessment and cross-dataset adaptation. Calibrated probabilities are converted into a transparent 0–100 risk index to support preventive outreach, psychosocial-support planning and resource-allocation decisions. The model is compared with baselines, including logistic regression, support vector machine, random forest, XGBoost, LightGBM, CatBoost, TabNet, FT–Transformer, NODE and DCN. Results show strong held-out performance with AUC = 0.885, average precision = 0.872, F1 = 0.808, Brier score = 0.145 and expected calibration error = 0.022, outperforming tested baselines. Five-fold robustness analysis produced a conservative mean test AUC of (Formula presented.), indicating moderate partition sensitivity. Key predictors include work interference, perceived stress, care access and support variables. Latent profiling identifies two behavioural subgroups with distinct risk patterns. After feature harmonization, target-domain adaptation and recalibration, external evaluation on an occupational burnout dataset achieves AUC = 0.941 and average precision = 0.936, supporting calibrated, interpretable and subgroup-aware decision support under dataset shift. © 2026 by the authors.

In [ ]:
DOI  =  '10.3390/bioengineering13060626'
df.loc[df['DOI'] == DOI]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
10,_0143,10.3390/bioengineering13060626,9.0,CALIBRATED DEEP LEARNING RISK INDEXING AND LAT...,Scopus,Occupational mental-health risk in knowledge-w...,104.0,0.0,143


In [ ]:
Query = 9
status = 0
criterio = 'O objetivo deste estudo é desenvolver e avaliar um framework de inteligência computacional para prever riscos à saúde mental no ambiente de trabalho, utilizando aprendizado de máquina interpretável, calibrado e adaptável a diferentes conjuntos de dados, com suporte à avaliação de equidade e à tomada de decisão em saúde pública.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.3390/bioengineering13060626,_0143,9.0,Scopus,104.0,CALIBRATED DEEP LEARNING RISK INDEXING AND LAT...,NaN,0,-1.0
1,10.3390/bioengineering13060626,_0143,9.0,Scopus,104.0,CALIBRATED DEEP LEARNING RISK INDEXING AND LAT...,O objetivo deste estudo é desenvolver e avalia...,0,0.0


### **Artigo 30**
```
Nome: A SYSTEMATIC REVIEW AND UNIFIED MULTI PERSPECTIVE CONCEPTUAL SYNTHESIS OF CONTEMPORARY MACHINE LEARNING PARADIGMS
```
Background: Machine learning (ML) has advanced many fields, enabling innovations once beyond technological reach. Self-driving cars, real-time language translation, personalized e-commerce, and sophisticated medical diagnostics exemplify its transformative potential. Although ML has become increasingly interdisciplinary, the existing literature often concentrates on individual paradigms or domains. ML cannot be confined to traditional tasks or architectures, given the rise of self-supervised learning, foundation models, federated learning, and growing ethical concerns about data privacy, algorithmic bias, fairness, interpretability, and sustainability. To address this gap, this systematic literature review (SLR) synthesizes contemporary ML developments through a structured, multi-perspective conceptual framework integrating representation, computational nature, learning paradigm, training philosophy, linearity, functional objectives, abstraction layers, hybridization, and special-purpose modeling across application domains. Methodology: This review analyzes 81 peer-reviewed studies published between 2017 and 2024 across healthcare, finance, education, engineering, and infrastructure. A PRISMA-guided selection process with explicit inclusion and exclusion criteria and structured keyword-based search strategies was employed to ensure methodological rigor and transparency. Results: The review categorizes contemporary ML paradigms across multiple complementary perspectives, highlighting their diverse cross-domain applications. The findings demonstrate a progression from traditional supervised, unsupervised, and reinforcement learning toward advanced paradigms such as self-supervised learning, foundation models, federated learning, and hybrid systems. Cross-domain analysis reveals strong concentration in healthcare and finance, while other societal domains remain underexplored. The study identifies comparative strengths, structural limitations, and persistent challenges related to interpretability, fairness, scalability, generalizability, and responsible deployment. Conclusions: This SLR consolidates the evolving ML landscape by synthesizing multiple taxonomic perspectives within a unified framework and demonstrates how representation, computational structure, learning philosophy, abstraction level, and application context shape paradigm selection and deployment. The review highlights structural challenges, including the absence of consistent paradigm definitions, limited cross-paradigm comparisons, and insufficient empirical validation. Fairness, interpretability, privacy, scalability, and sustainability emerge as essential considerations for responsible ML development, particularly in healthcare and finance. Although limited by its timeframe and scope, the proposed taxonomy offers a coherent conceptual foundation and supports systematic, responsible, and theoretically grounded advancement of the field. Copyright 2026 Mahbubetal. Distributed under Creative Commons CC-BY 4.0 https://creativecommons.org/licenses/by/4.0/

In [ ]:
DOI  = '10.7717/peerj-cs.3905'
df.loc[df['DOI'] == DOI]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
11,_0144,10.7717/peerj-cs.3905,9.0,A SYSTEMATIC REVIEW AND UNIFIED MULTI PERSPECT...,Scopus,Background: Machine learning (ML) has advanced...,105.0,0.0,144


In [ ]:
Query = 9
status = 0
criterio = 'O objetivo deste estudo é revisar e organizar os principais paradigmas contemporâneos de aprendizado de máquina por meio de uma estrutura conceitual unificada, analisando suas aplicações, vantagens, limitações e desafios relacionados à interpretabilidade, equidade, privacidade, escalabilidade e uso responsável.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.7717/peerj-cs.3905,_0144,9.0,Scopus,105.0,A SYSTEMATIC REVIEW AND UNIFIED MULTI PERSPECT...,NaN,0,-1.0
1,10.7717/peerj-cs.3905,_0144,9.0,Scopus,105.0,A SYSTEMATIC REVIEW AND UNIFIED MULTI PERSPECT...,O objetivo deste estudo é revisar e organizar ...,0,0.0


### **Artigo 31**
```
Nome: AI NATIVE 6G NETWORKS: ADAPTIVE RESOURCE ALLOCATION THROUGH EDGE INTEGRATED DEEP REINFORCEMENT LEARNING
```
This paper investigates AI-Native 6G networks that integrate deep reinforcement learning (DRL) agents at the network edge to achieve adaptive, low-latency, and energy-efficient resource allocation. We present a conceptual architecture - Edge-Integrated DRL (EIDRL) - that combines lightweight on-device policies, edge-hosted value models, and a hierarchical orchestration layer that mediates model updates, inference placement, and safety constraints. The proposed framework explicitly addresses 6G-era characteristics: massive heterogeneous devices, semantic and sensing-aided communications, integration of sensing/communication/computation (ISCC), and stringent reliability/latency requirements. We detail the learning formulation (state, action, reward, multi-timescale update rules), introduce techniques for sample-efficient training (model-based rollouts, transfer from foundation WLAMs, and federated/split learning hybrids), and propose mechanisms for safety, fairness, and compute-aware scheduling. Through analytical arguments and representative numerical scenarios (multi-tenant edge slices, ISAC nodes, and mobile user clusters), we demonstrate that EIDRL reduces regret relative to static baselines, lowers average latency under load, and improves spectral and energy efficiency when the edge controller coordinates inference placement and action aggregation. Finally, we identify open research directions - privacy-preserving multi-agent coordination, certifiable robustness of learned policies, semantics-aware reward design, and operational considerations for O-RAN/OAI integration - thereby providing a focused research agenda for adaptive resource allocation in AI-native 6G systems. © 2025 IEEE.

In [ ]:
DOI  = '10.1109/IIPEM65914.2025.11548455'
df.loc[df['DOI'] == DOI]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
12,_0146,10.1109/IIPEM65914.2025.11548455,9.0,AI NATIVE 6G NETWORKS: ADAPTIVE RESOURCE ALLOC...,Scopus,This paper investigates AI-Native 6G networks ...,107.0,0.0,146


In [ ]:
Query = 9
status = 0
criterio = 'O objetivo deste artigo é propor uma arquitetura baseada em aprendizado por reforço profundo na borda da rede para otimizar, de forma adaptativa, eficiente e segura, a alocação de recursos em redes 6G nativas em IA, considerando requisitos de baixa latência, eficiência energética, equidade e confiabilidade.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/IIPEM65914.2025.11548455,_0146,9.0,Scopus,107.0,AI NATIVE 6G NETWORKS: ADAPTIVE RESOURCE ALLOC...,NaN,0,-1.0
1,10.1109/IIPEM65914.2025.11548455,_0146,9.0,Scopus,107.0,AI NATIVE 6G NETWORKS: ADAPTIVE RESOURCE ALLOC...,O objetivo deste artigo é propor uma arquitetu...,0,0.0


### **Artigo 32**
```
Nome: STUDENT ACTIVITY MONITORING USING HYBRID DEEP LEARNING TECHNIQUE DURING ONLINE EXAMINATIONS
```
In the post-pandemic era, educational institutions have shifted toward online education cum examinations. Many international universities also offer online degrees through massive open online courses (MOOCs). The students are highly motivated to pursue such a course/degree online. As a result, ensuring the integrity of online examinations and preventing academic misconduct have become crucial. Various educational institutions offer online examinations through software platforms, which are costly and require additional human resources to monitor. Student activity monitoring during online examinations is an evolving field of research with many challenges, such as data availability for training the model, low-cost device compatibility, and cost-effectiveness. This research presents a transfer learning mechanism to monitor the student to recognize academic misconduct and malpractice in their activities. This research aims to develop a generalized, cost-effective, adaptable model to recognize any student. Real-time data has been collected through a webcam, sliced into images, and investigated through pre-trained convolutional neural network models due to less availability of training data and improved generalization. The proposed system also identifies any unauthorized devices, such as mobile phones or the other person’s presence in the room, to build a better system. The system’s performance has been experimented with by collecting both seen and unseen data for generalization. Finally, human proctors are involved in the experimentation process to validate the significance of the proposed monitoring system statistically on the same video recorded during the examination. Experimentation shows that the proposed system (InceptionV3 model) through transfer learning is highly reliable and ensures fairness and integrity in the online examination. © 2026 Scrivener Publishing LLC.

In [ ]:
df.loc[df["Nome"].str.contains('STUDENT ACTIVITY MONITORING USING HYBRID DEEP',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
14,_0193,-1,9.0,STUDENT ACTIVITY MONITORING USING HYBRID DEEP ...,Scopus,"In the post-pandemic era, educational institut...",154.0,0.0,193


In [ ]:
Nome = 'STUDENT ACTIVITY MONITORING USING HYBRID DEEP'
Query = 9
status = 0
criterio = 'O objetivo deste estudo é desenvolver um modelo baseado em aprendizado por transferência para monitorar automaticamente estudantes durante provas online, detectando fraudes e comportamentos inadequados de forma generalizável, adaptável, confiável e de baixo custo.'
comparacao, resultado  = avaliacao_artigo_nome (resultado,Nome, status,criterio)
comparacao

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,-1,_0193,9.0,Scopus,154.0,STUDENT ACTIVITY MONITORING USING HYBRID DEEP ...,NaN,0,-1.0
1,-1,_0193,9.0,Scopus,154.0,STUDENT ACTIVITY MONITORING USING HYBRID DEEP ...,O objetivo deste estudo é desenvolver um model...,0,0.0


### **Artigo 33**
```
Nome: AI BIAS AND FAIRNESS: UNVEILING SOLUTIONS FOR ETHICAL ALGORITHMIC PRACTICES
```
Artificial intelligence (AI) systems have progressively obtruded upon decision-making, across sectors, from content recommendations to fundamental society functions such as criminal justice and healthcare. As technologies advance, though, the issue of bias in AI has become one of the most vital issues to address. Biases in algorithms tend to reinforce systemic disadvantage, causing discriminatory results in hiring, loan disbursements, and law enforcement. This chapter explores the multifarious nature of AI bias and fairness, discussing both the causes and actual-world impact of algorithmic prejudice. Through discussions of major subjects like F1 scores, bias and retrieval-augmented generation, algorithmic oppression, and transfer learning, the conversation develops a deep apprehension of how biases are introduced, measured, and counteracted throughout the AI life cycle. Specifically, it identifies cutting-edge methods for mitigating bias, such as explainable AI, differential privacy, and federated learning, as well as new practices of algorithmic greenlining and bias auditing frameworks. The chapter also emphasizes the imperative for international regulatory frameworks to shape AI development, promoting fairness, transparency, and accountability. With a focus on actionable strategies and recent progress, it provides a guide for scholars, practitioners, and policymakers to create a more equitable and ethical AI environment. In pursuing algorithmic fairness, this chapter demands a multi-faceted strategy toward countering bias and designing AI systems that serve the whole of society equally. © 2026 Walter de Gruyter GmbH, Berlin/Boston, Genthiner Straße 13, 10785 Berlin.

In [ ]:
DOI  = '10.1515/9783112214374-022'
df.loc[df['DOI'] == DOI]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
20,_0250,10.1515/9783112214374-022,9.0,AI BIAS AND FAIRNESS: UNVEILING SOLUTIONS FOR ...,Scopus,Artificial intelligence (AI) systems have prog...,211.0,0.0,250


In [ ]:
Query = 9
status = 1
criterio = 'O objetivo deste capítulo é analisar as causas e os impactos dos vieses em sistemas de inteligência artificial, revisando técnicas de mitigação e estratégias para promover maior equidade, transparência e responsabilidade no desenvolvimento e uso da IA.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1515/9783112214374-022,_0250,9.0,Scopus,211.0,AI BIAS AND FAIRNESS: UNVEILING SOLUTIONS FOR ...,NaN,0,-1.0
1,10.1515/9783112214374-022,_0250,9.0,Scopus,211.0,AI BIAS AND FAIRNESS: UNVEILING SOLUTIONS FOR ...,O objetivo deste capítulo é analisar as causas...,0,1.0


### **Artigo 34**
```
Nome: MULTILINGUAL NLP MODELS: A REVIEW OF TECHNIQUES, TOOLS, AND REAL WORLD USE CASES
```
This survey summarized rapid developments in multilingual NLP. This growth was accelerated by increased digital communication globally. It reviewed the twenty most important works and delved into key areas such as sentiment analysis, machine translation, and hate speech detection. Most of the progress was achieved through powerful pre-trained models such as GPT-4, XLM-R, LLAMA-2, among others, and cross-lingual transfer learning. However, major challenges still remain. These are issues with low-resource languages, tokenization issues, bias within datasets, and lack of model explainability. Inconsistencies in inter-lingual evaluation also make progress harder. The paper concludes by highlighting that future efforts must focus on considerations of fairness, resource efficiency, and linguistic diversity. Realization of systems that are fair and culturally aware is critical in ensuring that such systems will be useful across the globe and meet ethical standards in NLP. ©2026 IEEE.

In [ ]:
df.loc[df["Nome"].str.contains('MULTILINGUAL NLP MODELS: A REVIEW OF TECHNIQUES,',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
21,_0254,10.1109/IITCEE67948.2026.11394472,9.0,MULTILINGUAL NLP MODELS: A REVIEW OF TECHNIQUE...,Scopus,This survey summarized rapid developments in m...,215.0,0.0,254


In [ ]:
DOI = '10.1109/IITCEE67948.2026.11394472'
Query = 9
status = 0
criterio = 'O objetivo deste artigo é revisar os avanços recentes em processamento de linguagem natural multilíngue, destacando o papel dos modelos pré-treinados e do aprendizado por transferência entre idiomas, além dos desafios relacionados à equidade, interpretabilidade e suporte a idiomas com poucos recursos.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/IITCEE67948.2026.11394472,_0254,9.0,Scopus,215.0,MULTILINGUAL NLP MODELS: A REVIEW OF TECHNIQUE...,NaN,0,-1.0
1,10.1109/IITCEE67948.2026.11394472,_0254,9.0,Scopus,215.0,MULTILINGUAL NLP MODELS: A REVIEW OF TECHNIQUE...,O objetivo deste artigo é revisar os avanços r...,0,0.0


### **Artigo 35**
```
Nome: THE CHALLENGES OF IMPLEMENTING GANS
```
Generative Adversarial Networks (GANs) have revolutionized artificial intelligence, enabling breakthroughs in image synthesis, content generation, and scientific discovery. However, their implementation presents significant challenges, including high data requirements, training instability, mode collapse, and computational demands. This chapter addresses these obstacles and explores innovative strategies to enhance GAN performance and scalability. GAN often demand vast, diverse datasets, which are expensive and difficult to obtain. We propose solutions such as data augmentation, synthetic data generation transfer learning, and domain adaptation to optimize GAN training in low resource environments. We further address training instability due to adversarial dynamics with advanced loss functions, architectural modifications, and optimization techniques, including gradient penalty and spectral normalization. Mode collapse in which the generator fails to produce diverse outputs, remains a critical problem. This chapter reviews recent state-of-the-art techniques like minibatch discrimination, unrolled GANs, and hybrid architectures that integrate VAEs to improve output diversity and fidelity. We also discuss computational challenges through lightweight architectures, model compression, and the use of accelerators such as GPUs and TPUs. Ethical considerations in terms of biases in the training data are integral to the deployment of GANs. We propose fairness aware GANs and adversarial debiasing methods that would mitigate these risks for the development of ethical and fair AI applications. By synthesizing state-of-the-art research all practical insights, this chapter equips readers with the tools to overcome GAN implementation challenges, unlocking their transformative potential across diverse fields. © 2026 Scrivener Publishing LLC.

In [ ]:
df.loc[df["Nome"].str.contains('THE CHALLENGES OF IMPLEMENTING GANS',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
23,_0263,10.1002/9781394358212.ch33,9.0,THE CHALLENGES OF IMPLEMENTING GANS,Scopus,Generative Adversarial Networks (GANs) have re...,224.0,0.0,263


In [ ]:
DOI = '10.1002/9781394358212.ch33'
Query = 9
status = 1
criterio = 'O objetivo deste capítulo é revisar os principais desafios na implementação de redes adversariais generativas (GANs) e apresentar estratégias para melhorar seu desempenho, estabilidade, escalabilidade e equidade, incluindo aprendizado por transferência, adaptação de domínio e técnicas de mitigação de vieses.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1002/9781394358212.ch33,_0263,9.0,Scopus,224.0,THE CHALLENGES OF IMPLEMENTING GANS,NaN,0,-1.0
1,10.1002/9781394358212.ch33,_0263,9.0,Scopus,224.0,THE CHALLENGES OF IMPLEMENTING GANS,O objetivo deste capítulo é revisar os princip...,0,1.0


### **Artigo 36**
```
Nome:GENERATIVE ARTIFICIAL INTELLIGENCE IN MEDICINE EMERGING DIRECTIONS AND ETHICAL CHALLENGES
```
The accelerated evolution of Generative Artificial Intelligence (GAI) has led to significant transformations across all fields of activity; however, through its capabilities of analysis, modeling, and personalized simulation, it redefines the paradigms of the medical domain. This paper proposes a conceptual architecture designed for the generation of medical images that contain the morphological characteristics associated with the presence of glaucoma. Another complementary direction addressed in this study focuses on the ethical challenges related to patient data confidentiality, the validation of results provided by generative models, and the assurance of their fairness, transparency, and traceability. In this context, the integration of emerging technologies in the medical field must not only meet technical requirements but also ensure compliance with deontological standards and the regulations in force. © The Author(s), under exclusive license to Springer Nature Switzerland AG 2026.

In [ ]:
df.loc[df["Nome"].str.contains('GENERATIVE ARTIFICIAL INTELLIGENCE IN MEDICINE EMERGING',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
25,_0279,10.1007/978-3-032-24724-7_47,11.0,GENERATIVE ARTIFICIAL INTELLIGENCE IN MEDICINE...,Scopus,The accelerated evolution of Generative Artifi...,14.0,0.0,279


In [ ]:
DOI = '10.1007/978-3-032-24724-7_47'
Query = 11
status = 0
criterio = 'O objetivo deste estudo é propor uma arquitetura conceitual baseada em inteligência artificial generativa para gerar imagens médicas de glaucoma, considerando também aspectos éticos relacionados à privacidade dos dados, equidade, transparência, rastreabilidade e validação dos modelos.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1007/978-3-032-24724-7_47,_0279,11.0,Scopus,14.0,GENERATIVE ARTIFICIAL INTELLIGENCE IN MEDICINE...,NaN,0,-1.0
1,10.1007/978-3-032-24724-7_47,_0279,11.0,Scopus,14.0,GENERATIVE ARTIFICIAL INTELLIGENCE IN MEDICINE...,O objetivo deste estudo é propor uma arquitetu...,0,0.0


### **Artigo 37**
```
Nome: CROSS MODALITY IMAGE SYNTHESIS IN VISION CARE USING GENERATIVE AI
```
The use of AI in vision care is revolutionizing the field by cross-modality image synthesis, which enhances medical imaging, improves diagnostic accuracy, and enables advanced treatment methods. This paper looks into the use of various generative AI models, such as VAEs, diffusion models, and GANs, for high-quality cross-modality medical images. Applications highlighted in the research include telescopic imaging using OCT, OCT image synthesis from fluorescein angiography, and multi-modal image enhancement in fundus photography. Using AI cross-modality image synthesis, detailed retinal images can be produced without the need to resort to invasive imaging procedures, leading to more precise diagnostic accuracy. Along with cases of sparse data, model generalization, and low adoption from clinicians, this study also looks into the forms of ethical governance that need to be considered, such as explainable AI validation cases, which require edifying examinations. It continues to be a challenge to create fairness and transparency measures for images generated by AI, especially for medical images, which calls for a combination of stakeholders like regulators, medical personnel, and AI builders. The text also points out the lack of standardized validation frameworks, benchmarking datasets, and clinical confidence, which model dependability relies on from the starting point. The study wraps up with a discussion of possible future developments as improving the designs of AI systems, creating comprehensive multimodal datasets, integrating imaging-based AI tools for image capturing and image processing treatment optimization in ophthalmology, Enhanced exploration is warranted for possibilities of real-time AI-driven diagnostics, privacy-preserving model training via federated learning, and decision support AI systems for ophthalmologists facing intricate case analysis challenges. © 2026 Scrivener Publishing LLC.

In [ ]:
df.loc[df["Nome"].str.contains('CROSS MODALITY IMAGE SYNTHESIS IN VISION CARE USIN',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
26,_0293,10.1002/9781394358212.ch3,11.0,CROSS MODALITY IMAGE SYNTHESIS IN VISION CARE ...,Scopus,The use of AI in vision care is revolutionizin...,28.0,0.0,293


In [ ]:
DOI = '10.1002/9781394358212.ch3'
Query = 11
status = 0
criterio = 'O objetivo deste estudo é revisar o uso de modelos de IA generativa para síntese de imagens médicas multimodais em oftalmologia, avaliando suas aplicações, desafios técnicos e éticos, e apontando direções para tornar esses sistemas mais confiáveis, justos e aplicáveis na prática clínica.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1002/9781394358212.ch3,_0293,11.0,Scopus,28.0,CROSS MODALITY IMAGE SYNTHESIS IN VISION CARE ...,NaN,0,-1.0
1,10.1002/9781394358212.ch3,_0293,11.0,Scopus,28.0,CROSS MODALITY IMAGE SYNTHESIS IN VISION CARE ...,O objetivo deste estudo é revisar o uso de mod...,0,0.0


### **Artigo 38**
```
Nome: ORDPRUNE KD: AN ORDINAL CONSISTENCY BASED MODEL COMPRESSION FRAMEWORK FOR DIABETIC RETINOPATHY GRADING
```
This study proposes OrdPrune-KD, an ordinal-consistency-driven model compression framework that integrates grade-aware structured pruning with Earth Mover’s Distance (EMD)-based knowledge distillation for diabetic retinopathy (DR) grading. Unlike conventional approaches that only consider ordinal relationships at the loss level, the proposed method incorporates ordinal priors into both model compression and knowledge transfer stages. Extensive experiments on APTOS 2019, Messidor-2, and IDRiD demonstrate that the proposed framework achieves a favorable balance between model compactness and predictive performance. In particular, under a 77% parameter reduction, the student model achieves competitive performance relative to the teacher model in terms of QWK while maintaining strong high-risk sensitivity. Additional ablation studies and fairness-controlled comparisons confirm that the performance gains are primarily attributed to the proposed ordinal-aware design rather than output formulation differences. These results indicate that OrdPrune-KD provides an effective and deployable solution for lightweight DR grading systems. © 2026 by the authors.

In [ ]:
df.loc[df["Nome"].str.contains('ORDPRUNE KD: AN ORDINAL CONSISTENCY BASED MODEL COMPRESSION',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
32,_0323,10.3390/s26123636,11.0,ORDPRUNE KD: AN ORDINAL CONSISTENCY BASED MODE...,Scopus,"This study proposes OrdPrune-KD, an ordinal-co...",58.0,0.0,323


In [ ]:
DOI = '10.3390/s26123636'
Query = 11
status = 0
criterio = 'O objetivo deste estudo é propor um método de compressão de modelos para classificação da retinopatia diabética que preserve as relações ordinais entre as classes, reduzindo o tamanho do modelo sem comprometer o desempenho e a sensibilidade na detecção de casos de alto risco.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.3390/s26123636,_0323,11.0,Scopus,58.0,ORDPRUNE KD: AN ORDINAL CONSISTENCY BASED MODE...,NaN,0,-1.0
1,10.3390/s26123636,_0323,11.0,Scopus,58.0,ORDPRUNE KD: AN ORDINAL CONSISTENCY BASED MODE...,O objetivo deste estudo é propor um método de ...,0,0.0


### **Artigo 39**
```
Nome: HOW IS BIAS LEARNED IN MEDICAL IMAGE ANALYSIS MODELS? AN EXPLORATION OF THE ENCODING OF DEMOGRAPHIC INFORMATION IN DEEP LEARNING MODELS TRAINED TO DETECT ABNORMALITIES ON CHEST X RAYS
```
Deep learning models achieve strong diagnostic performance in medical imaging, yet often exhibit systematic performance disparities across demographic subgroups. Although prior work has shown that attributes such as age, sex and race are encoded within internal representations, it remains unclear how the structure of these representations contributes to subgroup-level differences in prediction behaviour. This study aims to examine how demographic information is embedded in chest X-ray classifiers and how latent-space structure relates to observed sensitivity disparities. We analysed two large-scale chest X-ray datasets, CheXpert and MIMIC-CXR, using DenseNet-121 models trained for multi-label disease classification. In addition to standard output-level evaluation, we conducted representation-level analyses using linear probes, embedding statistics and geometric measures to characterise subgroup differences in activation strength, latent-space proximity and model confidence. Disparities were assessed across age, race and sex by jointly examining feature encodings, logits, energy scores and true-positive rates. Demographic attributes showed limited direct association with disease labels and low standalone predictive utility, yet were strongly encoded within internal features. Younger and Black/African American patients consistently exhibited higher feature norms, greater separation in latent space and lower joint logit energy, despite comparable overall discrimination performance. These representational patterns persisted after accounting for label configuration and were associated with larger sensitivity gaps, consistent with structural suppression in which certain subgroups occupy sparser, lower-activation regions of the representation space. Sex-based differences were comparatively modest across representational and performance metrics. Subgroup disparities in chest X-ray classification are closely linked to how demographic groups are positioned and activated within latent space, rather than to directional misalignment alone. Representation-level diagnostics based on activation magnitude, density and energy provide mechanistic insight into model behaviour and highlight limitations of mitigation strategies that focus solely on feature removal or post hoc thresholding. These findings support the use of representation-level analysis as a principled component of fairness evaluation and mitigation design in clinical AI systems. © The Author(s) 2026.

In [ ]:
df.loc[df["Nome"].str.contains('HOW IS BIAS LEARNED IN MEDICAL IMAGE ANALYSIS MODELS?',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
34,_0346,10.1007/s10278-026-02073-0,11.0,HOW IS BIAS LEARNED IN MEDICAL IMAGE ANALYSIS ...,Scopus,Deep learning models achieve strong diagnostic...,81.0,0.0,346


In [ ]:
DOI = '10.1007/s10278-026-02073-0'
Query = 11
status = 1
criterio = 'O objetivo deste estudo é investigar como informações demográficas são representadas no espaço latente de modelos de aprendizado profundo para classificação de radiografias de tórax e como essa representação contribui para diferenças de desempenho entre subgrupos demográficos, apoiando avaliações mais robustas de equidade em IA clínica.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1007/s10278-026-02073-0,_0346,11.0,Scopus,81.0,HOW IS BIAS LEARNED IN MEDICAL IMAGE ANALYSIS ...,NaN,0,-1.0
1,10.1007/s10278-026-02073-0,_0346,11.0,Scopus,81.0,HOW IS BIAS LEARNED IN MEDICAL IMAGE ANALYSIS ...,O objetivo deste estudo é investigar como info...,0,1.0


### **Artigo 40**
```
Nome: FEDERATED LEARNING WITH LAYER WISE CONTRIBUTION ESTIMATION AND CONSENSUS BASED PARAMETER PERSONALIZATION FOR MEDICAL IMAGE CLASSIFICATION
```
Personalized Federated Learning (PFL) presents a promising paradigm for mitigating privacy concerns and addressing data heterogeneity in medical images. However, the presence of data heterogeneity can introduce bias into the global model. As a result, the biased global model may degrade local performance in the personalization phase. To tackle this challenge, we propose FedLP, a novel framework that enhances global aggregation fairness and improves personalized performance. First, we propose a layer-wise aggregation strategy on the server, where it dynamically estimates each client's contribution based on the gradient discrepancies. Consider the aggregated global model may not consistently align with local distributions, we then introduce an adaptive parameter personalization process on the client side. By selectively instituting local parameters with the global model parameter with maximizing consensus optimization, clients can better retain global knowledge they need and maintain important model parameters to local data distributions. Experimental results demonstrate that FedLP outperforms state-of-the-art methods and achieves fairer performance for several medical image classification tasks. © 2026 IEEE.

In [ ]:
df.loc[df["Nome"].str.contains('FEDERATED LEARNING WITH LAYER WISE CONTRIBUTION ESTIMATION',na=False)]


,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
35,_0368,10.1109/ISBI61048.2026.11515338,11.0,FEDERATED LEARNING WITH LAYER WISE CONTRIBUTIO...,Scopus,Personalized Federated Learning (PFL) presents...,103.0,0.0,368


In [ ]:
DOI = '10.1109/ISBI61048.2026.11515338'
Query = 11
status = 0
criterio = 'O objetivo deste estudo é propor um método de aprendizado federado personalizado para classificação de imagens médicas que reduza os efeitos da heterogeneidade dos dados, promovendo uma agregação global mais justa e melhor desempenho dos modelos locais.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/ISBI61048.2026.11515338,_0368,11.0,Scopus,103.0,FEDERATED LEARNING WITH LAYER WISE CONTRIBUTIO...,NaN,0,-1.0
1,10.1109/ISBI61048.2026.11515338,_0368,11.0,Scopus,103.0,FEDERATED LEARNING WITH LAYER WISE CONTRIBUTIO...,O objetivo deste estudo é propor um método de ...,0,0.0


### **Artigo 41**
```
Nome: DERMAVIGNET: A HYBRID VISION TRANSFORMER AND CNN BASED FRAMEWORK WITH EXPLAINABLE AI FOR ROBUST SKIN DISEASE CLASSIFICATION
```
Accurate diagnosis of skin diseases is essential for fair healthcare, but rare conditions are often underrepresented in existing dermatological datasets. This imbalance makes them more difficult to classify reliably. To address the challenge, we propose DermaViGNet, a hybrid deep learning model that integrates Vision Transformer (ViT) and VGG16. A dataset of 9,548 dermatoscopic images collected from hospital and online sources is used for training and validation. The architecture combines convolutional networks to capture local texture features with transformers to learn global context. To improve interpretability, Local Interpretable Model-Agnostic Explanation (LIME) highlights image regions most influential to predictions, providing transparency for clinical use. DermaViGNet achieved 98% accuracy with a validation loss of 0.073, outperforming established CNN- and Transformer-based baselines. These findings show that incorporating diverse conditions into AI diagnostic systems can enhance fairness, explainability, and performance in medical image analysis. © 2025 IEEE.

In [ ]:
df.loc[df["Nome"].str.contains('DERMAVIGNET: A HYBRID VISION TRANSFORMER AND CNN BASED',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
37,_0388,10.1109/ICCIT68739.2025.11490111,11.0,DERMAVIGNET: A HYBRID VISION TRANSFORMER AND C...,Scopus,Accurate diagnosis of skin diseases is essenti...,123.0,0.0,388


In [ ]:
DOI = '10.1109/ICCIT68739.2025.11490111'
Query = 11
status = 1
criterio = 'O objetivo deste estudo é desenvolver um modelo híbrido de aprendizado profundo para diagnóstico de doenças de pele que melhore a classificação de condições raras, aumentando o desempenho, a interpretabilidade e a equidade em sistemas de análise de imagens médicas.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/ICCIT68739.2025.11490111,_0388,11.0,Scopus,123.0,DERMAVIGNET: A HYBRID VISION TRANSFORMER AND C...,NaN,0,-1.0
1,10.1109/ICCIT68739.2025.11490111,_0388,11.0,Scopus,123.0,DERMAVIGNET: A HYBRID VISION TRANSFORMER AND C...,O objetivo deste estudo é desenvolver um model...,0,1.0


### **Artigo 42**
```
Nome: INTELLIGENT DIAGNOSIS: LEVERAGING ARTIFICIAL INTELLIGENCE TO DETECT AND MANAGE INFECTIOUS AND NONINFECTIOUS DISEASES
```
Infectious and noninfectious diseases remain major global health threats, requiring rapid, accurate, and innovative approaches for diagnosis, surveillance, and management. Artificial Intelligence (AI), particularly Machine Learning (ML) and Deep Learning (DL), has emerged as a powerful catalyst in strengthening disease detection, early outbreak warning, contact tracing, and drug discovery. AI-driven platforms provide real-time insights for vaccine development, prediction of structural proteins, and identification of therapeutic targets, significantly accelerating responses to emerging pathogens. Generative Artificial Intelligence (GenAI) represents a recent breakthrough in healthcare, capable of producing synthetic data, medical images, and clinical text that can assist in diagnosis, enhance clinical decision-making, and improve patient outcomes. Despite its promise, the effective integration of GenAI into routine healthcare is challenged by workforce readiness, medicolegal concerns, ethical implications, and complexities of service delivery. ML and DL continue to revolutionize clinical workflows through applications such as smart electronic health records, medical image interpretation, disease classification, risk prediction, and optimization of clinical trials. DL modelsâ€”especially Convolutional Neural Networksâ€”excel in processing radiological images including CT scans, X-rays, MRIs (magnetic resonance imaging), and ultrasound, enabling accurate differentiation between normal and pathological conditions. However, AI deployment faces persistent barriers related to data privacy, sensitivity of patient information, interoperability, and the computational demands of large neural networks. Integrating AI with mobile sensing technologies and the Internet of Things (IoT) further increases implementation challenges. Ensuring transparency, fairness, and protection against algorithmic bias is essential to prevent discrimination and maintain trust. Overall, responsible and strategic adoption of AI technologies can significantly strengthen healthcare systems, improve diagnostic accuracy, and better address the needs of an expanding global population. Â© The Editor(s) (if applicable) and The Author(s), under exclusive license to Springer Nature Singapore Pte Ltd. 2026.

In [ ]:
df.loc[df["Nome"].str.contains('INTELLIGENT DIAGNOSIS: LEVERAGING ARTIFICIAL',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
38,_0389,10.1007/978-981-95-9027-8_7,11.0,INTELLIGENT DIAGNOSIS: LEVERAGING ARTIFICIAL I...,Scopus,Infectious and noninfectious diseases remain m...,124.0,0.0,389


In [ ]:
DOI = '10.1007/978-981-95-9027-8_7'
Query = 11
status = 1
criterio = 'O objetivo deste estudo é revisar o papel da inteligência artificial, incluindo aprendizado de máquina, aprendizado profundo e IA generativa, no diagnóstico, vigilância e tratamento de doenças, discutindo suas aplicações, benefícios e desafios relacionados à privacidade, equidade, transparência e implementação na saúde.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1007/978-981-95-9027-8_7,_0389,11.0,Scopus,124.0,INTELLIGENT DIAGNOSIS: LEVERAGING ARTIFICIAL I...,NaN,0,-1.0
1,10.1007/978-981-95-9027-8_7,_0389,11.0,Scopus,124.0,INTELLIGENT DIAGNOSIS: LEVERAGING ARTIFICIAL I...,O objetivo deste estudo é revisar o papel da i...,0,1.0


### **Artigo 43**
```
Nome: FAIRNESS AND AI GENERALIZABILITY IN MEDICAL IMAGE ANALYSIS
```
The rapid evolution of artificial intelligence (AI) and in particular machine learning (ML) for healthcare applications has opened many new exciting opportunities to automatically analyze increasingly complex clinical data, especially related to medical imaging, one of the biggest data contributors in health care. However, while machine learning models have demonstrated considerable potential in the research setting to improve and accelerate diagnostic accuracy, reduce clinician workload, and enable integration of multi-modal data, their real-world deployment in clinical settings remains limited in many cases. A major obstacle is that ML models trained on medical images consistently fail to generalize well and perform poorly when applied to data from institutions, scanners, and patient populations that were not or poorly represented in the training set. These performance gaps often stem from biological and non-biological variations and biases that are only spuriously but not causally correlated with the medical task of interest. However, it still remains an open question how such biases propagate through deep learning model architectures and shape learned representations. This invited paper summarizes our recent efforts to address these challenges through controlled bias experiments and distributed learning methods. More precisely, the Simulated Bias in Artificial Medical Images (SimBA) framework is introduced, which enables the generation of realistic brain MRI datasets with known and fully controllable morphological and intensity-based biases, thereby facilitating counterfactual analyses of how biases are encoded and used by deep learning models. Using SimBA, we demonstrate that standard convolutional neural networks encode biases across all layers and that shortcut learning depends on various factors, including spatial proximity, bias salience, and class prevalence. We further discuss distributed training strategies as a scalable solution to overcome structural barriers to data sharing and enhance deep learning model generalizability. Together, this work provides foundational insights toward developing robust, interpretable, and equitable ML solutions for the analysis of medical imaging data and downstream computer-aided diagnosis tasks. © 2026 SPIE. All rights reserved.

In [ ]:
df.loc[df["Nome"].str.contains('FAIRNESS AND AI GENERALIZABILITY IN MEDICAL ',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
39,_0407,10.1117/12.3109369,11.0,FAIRNESS AND AI GENERALIZABILITY IN MEDICAL IM...,Scopus,The rapid evolution of artificial intelligence...,142.0,0.0,407


In [ ]:
DOI = '10.1117/12.3109369'
Query = 11
status = 1
criterio = 'O objetivo deste estudo é investigar como vieses são aprendidos e propagados em modelos de aprendizado profundo para imagens médicas, propondo um framework para análise controlada desses vieses e explorando estratégias de aprendizado distribuído para melhorar a generalização, interpretabilidade e equidade dos modelos.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1117/12.3109369,_0407,11.0,Scopus,142.0,FAIRNESS AND AI GENERALIZABILITY IN MEDICAL IM...,NaN,0,-1.0
1,10.1117/12.3109369,_0407,11.0,Scopus,142.0,FAIRNESS AND AI GENERALIZABILITY IN MEDICAL IM...,O objetivo deste estudo é investigar como vies...,0,1.0


### **Artigo 44**
```
Nome: ADVANCEMENTS IN IMAGE CAPTIONING: A COMPREHENSIVE SURVEY ON TECHNIQUES, MODALITIES, AND APPLICATIONS
```
Recent advances in image captioning come from deep learning methods. These methods connect visual understanding with natural language generation. Early CNN- RNN models had some success, but they struggled with making sense and reasoning in context. To fix these issues, researchers added multi-attention mechanisms, hierarchical designs, and multimodal pre-training. These changes greatly improve caption quality. In medicine, models like MedViLL use BERT designs to improve performance on various tasks. Also, using patient background information shows the value of specific medical knowledge. Generative adversarial and knowledge-enhanced frameworks, like RAGAN and EURAIC, improve feature representation. Also, fine-grained techniques such as ASP help achieve a deeper semantic understanding. This paper highlights recent developments that show the need to combine vision and language models. It shows we need more research on common-sense reasoning, fairness, and real-time captioning. These areas will help create more human-like and context-aware image descriptions. © 2025 IEEE.

In [ ]:
df.loc[df["Nome"].str.contains('ADVANCEMENTS IN IMAGE CAPTIONING: A COMPREHENSIVE SURVEY',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
40,_0412,10.1109/DASA68193.2025.11499090,11.0,ADVANCEMENTS IN IMAGE CAPTIONING: A COMPREHENS...,Scopus,Recent advances in image captioning come from ...,147.0,0.0,412


In [ ]:
DOI = '10.1109/DASA68193.2025.11499090'
Query = 11
status = 0
criterio = 'O objetivo deste artigo é revisar os avanços recentes em geração automática de descrições de imagens, destacando a integração entre modelos de visão e linguagem, suas aplicações em imagens médicas e os desafios futuros relacionados ao raciocínio, equidade e geração de descrições em tempo real.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/DASA68193.2025.11499090,_0412,11.0,Scopus,147.0,ADVANCEMENTS IN IMAGE CAPTIONING: A COMPREHENS...,NaN,0,-1.0
1,10.1109/DASA68193.2025.11499090,_0412,11.0,Scopus,147.0,ADVANCEMENTS IN IMAGE CAPTIONING: A COMPREHENS...,O objetivo deste artigo é revisar os avanços r...,0,0.0


### **Artigo 45**
```
Nome: FAIRVLM: ENHANCING FAIRNESS AND PROMPT SENSITIVITY IN VISION LANGUAGE MODELS FOR MEDICAL IMAGE SEGMENTATION
```
Vision-language models (VLMs) have demonstrated substantial promise in medical image segmentation by utilizing radiology reports as prompts to segment regions of interest. However, VLM deployment in clinical settings is challenged by two intertwined issues: i) demographic bias, where performance varies across demographic groups, and ii) prompt sensitivity, where semantically similar prompts yield inconsistent outputs. These challenges are interconnected; demographic underrepresentation can worsen a model's sensitivity to prompts, and prompt instability can more heavily affect certain demographic groups. In this study, we present FairVLM, a unified framework that addresses both demographic disparity and prompt sensitivity in VLMs. FairVLM integrates three key components: (1) Semantic-Retaining Counterfactual Prompting (SRCP), which generates clinically consistent and diverse prompt variations via large language models; (2) Demographic-Aware Feature Normalization (DAFN), a lightweight module that mitigates latent representation bias across demographic groups; and (3) a Fairness-Calibrated Loss (FCL) that explicitly penalizes performance disparities while encouraging prompt consistency. Extensive evaluations on the Harvard-FairSeg dataset show that FairVLM significantly improves equity-scaled segmentation. It also reduces demographic disparity (DI) by over 65% and relative performance gap (RPG) by over 60%, while maintaining or boosting overall accuracy. FairVLM is robust to prompt changes, with less than 0.5% performance drop across varied prompts, and also generalizes well on unseen datasets. These findings present FairVLM as a new state-of-the-art, robust, and adaptable framework for a fair, prompt-invariant vision-language model. Code and data are available at https://github.com/Rahman-Motiur/FairVLM. © 2026 IEEE.

In [ ]:
df.loc[df["Nome"].str.contains('FAIRVLM: ENHANCING FAIRNESS AND PROMPT SENSITIVITY IN VISIO',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
43,_0424,10.1109/WACV61042.2026.00719,11.0,FAIRVLM: ENHANCING FAIRNESS AND PROMPT SENSITI...,Scopus,Vision-language models (VLMs) have demonstrate...,159.0,0.0,424


In [ ]:
DOI = '10.1109/WACV61042.2026.00719'
Query = 11
status = 1
criterio = 'O objetivo deste estudo é desenvolver um modelo de visão e linguagem para segmentação de imagens médicas que reduza vieses demográficos e a sensibilidade às variações dos prompts, promovendo maior equidade, robustez e capacidade de generalização.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/WACV61042.2026.00719,_0424,11.0,Scopus,159.0,FAIRVLM: ENHANCING FAIRNESS AND PROMPT SENSITI...,NaN,0,-1.0
1,10.1109/WACV61042.2026.00719,_0424,11.0,Scopus,159.0,FAIRVLM: ENHANCING FAIRNESS AND PROMPT SENSITI...,O objetivo deste estudo é desenvolver um model...,0,1.0


### **Artigo 46**
```
Nome: LEARNING TO PREDICT MENOPAUSAL HEALTH RISK TRAJECTORIES USING DEEP LEARNING
```
Long term outcomes in terms of health are profoundly impacted by the complex physiological transition of menopause which is characterised by changes in the vasomotor, metabolic, psychological and genitourinary systems. Due to broken healthcare system, unconsistent symptom reporting and delayed clinical evaluation, early identification of risk trajectory is still limited. This study is a summation of the developments in menopausal and postmenopausal women risk for Health by the deep learning approaches. Deep learning models have the capacity to boost significantly the ability to detect and predict the occurrence of significant conditions such as cardiovascular disease, osteoporosis, breast cancer and mood disorders based on evidence from biomarker studies, medical imaging, electronic health records, wearable sensor data, and natural language processing. Additionally, these models enable the risk profile to be nailed down with precision, as well as real-time symptom tracking and monitoring. Deep Learning - daily digital platforms are also supporting people to better accessibility of remote care, lifestyle modification advice, and customized therapeutic planning. Although the results prove the potential for deep learning to revolutionize menopausal care from a reactive to a proactive model, issues such as algorithmic fairness, data privacy, model transparency and clinical integration remain. The safe, equitable and scalable application of deep learning systems based on menopausal health management require these obstacles to be overcome with interdisciplinary co-operation and an ethical governance. © 2026 IEEE.

In [ ]:
df.loc[df["Nome"].str.contains('LEARNING TO PREDICT MENOPAUSAL HEALTH RISK',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
44,_0441,10.1109/ICAISS68683.2026.11526437,11.0,LEARNING TO PREDICT MENOPAUSAL HEALTH RISK TRA...,Scopus,Long term outcomes in terms of health are prof...,176.0,0.0,441


In [ ]:
DOI = '10.1109/ICAISS68683.2026.11526437'
Query = 11
status = 0
criterio = 'O objetivo deste estudo é revisar os avanços do aprendizado profundo na predição e monitoramento de riscos à saúde de mulheres na menopausa e pós-menopausa, destacando suas aplicações, benefícios e desafios relacionados à equidade, privacidade, transparência e integração clínica.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1109/ICAISS68683.2026.11526437,_0441,11.0,Scopus,176.0,LEARNING TO PREDICT MENOPAUSAL HEALTH RISK TRA...,NaN,0,-1.0
1,10.1109/ICAISS68683.2026.11526437,_0441,11.0,Scopus,176.0,LEARNING TO PREDICT MENOPAUSAL HEALTH RISK TRA...,O objetivo deste estudo é revisar os avanços d...,0,0.0


### **Artigo 47**
```
Nome: FAIRGEN: PREFERENCE ALIGNED DIFFUSION FOR DEMOGRAPHICALLY EQUITABLE MEDICAL IMAGE SYNTHESIS
```
Medical imaging is central to modern diagnostics, and artificial intelligence (AI) systems are increasingly used to support image-based analysis by improving efficiency, accuracy, and access to care. However, inequities in healthcare access and differential disease prevalence create severe demographic imbalances in clinical image data. Such imbalances are compounded by the fact that diseases can manifest with distinct features across demographic groups, rendering certain phenotypic presentations naturally rare. AI models trained on such imbalanced data risk perpetuating diagnostic bias and widening healthcare disparities. Here we introduce FairGen, a fairness-aware diffusion framework that synthesizes demographically balanced medical images while preserving pathology-relevant visual features. By embedding physician-aligned preferences into the generation process, FairGen improves subgroup coverage during synthesis and downstream classification. Applied to dermatology, radiology, and neuroimaging benchmark tasks, FairGen achieves fairness improvements of 95.9% for skin images, 80.0% for chest radiography, and 35.2% for brain MRI, while maintaining competitive diagnostic accuracy relative to models trained on original clinical data. Clinician-facing expert review and external validation on independent cohorts further support that these gains extend beyond standard fidelity metrics and are not confined to the original in-distribution datasets.

In [ ]:
df.loc[df["Nome"].str.contains('FAIRGEN: PREFERENCE ALIGNED DIFFUSION FOR',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
47,_1067,10.1038/s41746-026-02868-z,11.0,FAIRGEN: PREFERENCE ALIGNED DIFFUSION FOR DEMO...,PubMed,Author information:\n(1)Swanson School of Engi...,14.0,1.0,1067


In [ ]:
DOI = '10.1038/s41746-026-02868-z'
Query = 11
status = 1
criterio = 'O objetivo deste estudo é desenvolver um modelo de difusão sensível à equidade para gerar imagens médicas sintéticas demograficamente balanceadas, preservando características relevantes da patologia e reduzindo vieses em modelos de diagnóstico por IA.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.1038/s41746-026-02868-z,_1067,11.0,PubMed,14.0,FAIRGEN: PREFERENCE ALIGNED DIFFUSION FOR DEMO...,NaN,0,-1.0
1,10.1038/s41746-026-02868-z,_1067,11.0,PubMed,14.0,FAIRGEN: PREFERENCE ALIGNED DIFFUSION FOR DEMO...,O objetivo deste estudo é desenvolver um model...,0,1.0


### **Artigo 48**
```
Nome: ETHICAL IMPLICATIONS OF THE USE OF AI BASED TECHNOLOGIES FOR MEDICAL IMAGE CLASSIFICATION SYSTEMS IN SCREENING: A QUALITATIVE SYSTEMATIC REVIEW
```
BACKGROUND: The integration of artificial intelligence in medical image
classification for screening has the potential to enhance efficiency, diagnostic
accuracy and accessibility. However, ethical concerns such as accountability,
bias, transparency and the impact on healthcare professionals remain critical.
This review synthesises qualitative evidence on the ethical considerations
surrounding artificial intelligence adoption in screening programmes.
METHODS: A systematic search of qualitative studies, from June 2020 to September
2024, was conducted across multiple databases: MEDLINE, EMBASE, PsycInfo®
(American Psychological Association, Washington, DC, USA) and Cumulative Index
to Nursing and Allied Health Literature. Primary qualitative studies exploring
healthcare professionals', patients' and other stakeholders' perspectives on
artificial intelligence in screening were included. Thematic analysis was
performed, and findings were assessed using the Grading of Recommendations
Assessment, Development and Evaluation-Confidence in the Evidence from Reviews
of Qualitative Research approach to evaluate confidence in the evidence.
RESULTS: Fourteen qualitative studies were included, covering perspectives from
clinicians, radiologists, artificial intelligence developers, policy-makers and
patients. Key ethical concerns identified included: (1) the necessity of human
oversight to ensure that artificial intelligences diagnostic recommendations are
appropriate; (2) challenges in assigning liability when artificial intelligence
errors occur; (3) risks of algorithmic bias due to discrepancies between
training data sets and real-world populations; (4) concerns over data privacy,
cybersecurity and informed consent in artificial intelligence-driven
decision-making; (5) the need for transparency in artificial intelligence
decision-making processes to build trust and (6) potential deskilling of
healthcare professionals and shifts in professional responsibilities. While
artificial intelligence was seen as a valuable tool to augment clinical
decision-making, stakeholders emphasised that ethical frameworks must guide its
implementation to maintain public trust and patient safety.
CONCLUSION: This review highlights the critical considerations that must be
addressed to ensure the responsible integration of artificial intelligence in
medical screening. Policy-makers, healthcare institutions and developers should
prioritise human oversight, robust regulatory frameworks and strategies to
mitigate bias and ensure transparency. Future research should focus on
disease-specific artificial intelligence applications and long-term ethical
implications.
STUDY REGISTRATION: The protocol for this study is registered on PROSPERO as
CRD42024599536.
FUNDING: This award was funded by the National Institute for Health and Care
Research (NIHR) Evidence Synthesis programme (NIHR award ref: NIHR172233) and is
published in full in Health Technology Assessment; Vol. 30, No. 51. See the NIHR
Funding and Awards website for further award information.

In [ ]:
df.loc[df["Nome"].str.contains('ETHICAL IMPLICATIONS OF THE USE OF AI BASED T',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
48,_1062,10.3310/GJSE4912,11.0,ETHICAL IMPLICATIONS OF THE USE OF AI BASED TE...,PubMed,BACKGROUND: The integration of artificial inte...,30.0,1.0,1062


In [ ]:
DOI = '10.3310/GJSE4912'
Query = 11
status = 1
criterio = 'O objetivo desta revisão sistemática é sintetizar evidências qualitativas sobre as implicações éticas do uso da inteligência artificial na classificação de imagens médicas para rastreamento, com foco em vieses, transparência, responsabilidade, privacidade e supervisão humana.'
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.3310/GJSE4912,_1062,11.0,PubMed,30.0,ETHICAL IMPLICATIONS OF THE USE OF AI BASED TE...,NaN,0,-1.0
1,10.3310/GJSE4912,_1062,11.0,PubMed,30.0,ETHICAL IMPLICATIONS OF THE USE OF AI BASED TE...,O objetivo desta revisão sistemática é sinteti...,0,1.0


### **Artigo 49**
```
Nome: EQUITABLE HEALTH INTELLIGENCE: AN OPEN BENCHMARK OF MULTI POPULATION MACHINE LEARNING FOR OMICS BASED CANCER PROGNOSIS
```

In [ ]:
df.loc[df["Nome"].str.contains('EQUITABLE HEALTH INTELLIGENCE: AN OPEN BENCHMARK',na=False)]

,Codigo,DOI,Query,Nome,Base,Abstract,Artigo,Cod,Codigo_Anteriores
46,_1033,10.64898/2026.05.29.728755,9.0,EQUITABLE HEALTH INTELLIGENCE: AN OPEN BENCHMA...,Pubmed,NaN,24.0,NaN,1033


In [ ]:
DOI  = '10.64898/2026.05.29.728755'
Query = 9
status = 1
criterio = 'Modelos treinados apenas em datasets europeus. '
print,resultado = avaliacao_artigo (resultado, DOI, status, criterio)
print

,DOI,Codigo,Query,Base,Artigo,Nome,Criterio_Final,Repetidos,Status_Final
0,10.64898/2026.05.29.728755,_1033,9.0,Pubmed,24.0,EQUITABLE HEALTH INTELLIGENCE: AN OPEN BENCHMA...,NaN,0,-1.0
1,10.64898/2026.05.29.728755,_1033,9.0,Pubmed,24.0,EQUITABLE HEALTH INTELLIGENCE: AN OPEN BENCHMA...,Modelos treinados apenas em datasets europeus.,0,1.0


In [ ]:
resultado.Status_Final.value_counts()

,count
Status_Final,
0.0,292
1.0,169


# Step 2: Export Base Final

In [ ]:
resultado.to_csv(path_to + '5_Lista_Final_Artigos_Revisao_202607.csv', index=False)